In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/Celeb-DF'))

['List_of_testing_videos.txt', 'Celeb-synthesis', 'Celeb-real', 'YouTube-real', 'Frames-Real', 'Frames-Fake', 'Frames-Real-20240626T130257Z-001.zip', 'Frames-Fake-20240626T130258Z-001.zip', 'Frames-YouTube-real', 'model.h5', 'Split-Frames', 'model_checkpoint.h5']


In [ ]:
# Define the paths to your dataset in Google Drive
real_videos_dir = '/content/drive/MyDrive/Celeb-DF/Celeb-real'
fake_videos_dir = '/content/drive/MyDrive/Celeb-DF/Celeb-synthesis'

# Verify that the paths are correct
import os
print(os.listdir(real_videos_dir))
print(os.listdir(fake_videos_dir))

['id0_0000.mp4', 'id0_0001.mp4', 'id0_0002.mp4', 'id0_0005.mp4', 'id0_0006.mp4', 'id10_0000.mp4', 'id0_0009.mp4', 'id0_0003.mp4', 'id0_0008.mp4', 'id0_0007.mp4', 'id0_0004.mp4', 'id11_0005.mp4', 'id10_0001.mp4', 'id10_0006.mp4', 'id11_0006.mp4', 'id10_0002.mp4', 'id10_0008.mp4', 'id10_0007.mp4', 'id10_0003.mp4', 'id10_0009.mp4', 'id10_0005.mp4', 'id11_0003.mp4', 'id11_0002.mp4', 'id11_0000.mp4', 'id11_0001.mp4', 'id10_0004.mp4', 'id11_0004.mp4', 'id12_0004.mp4', 'id13_0005.mp4', 'id13_0008.mp4', 'id13_0000.mp4', 'id11_0008.mp4', 'id13_0007.mp4', 'id13_0009.mp4', 'id13_0006.mp4', 'id11_0010.mp4', 'id12_0000.mp4', 'id12_0001.mp4', 'id13_0004.mp4', 'id11_0009.mp4', 'id13_0001.mp4', 'id12_0003.mp4', 'id11_0007.mp4', 'id12_0006.mp4', 'id13_0003.mp4', 'id12_0005.mp4', 'id13_0002.mp4', 'id12_0002.mp4', 'id13_0015.mp4', 'id13_0014.mp4', 'id16_0003.mp4', 'id13_0013.mp4', 'id13_0010.mp4', 'id16_0000.mp4', 'id13_0012.mp4', 'id16_0001.mp4', 'id16_0002.mp4', 'id13_0011.mp4', 'id17_0006.mp4', 'id16_

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os
import cv2
# Output paths for frames
real_frames_path = '/content/drive/MyDrive/Celeb-DF/Frames-Real'
fake_frames_path = '/content/drive/MyDrive/Celeb-DF/Frames-Fake'

# First Pipeline

new added

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define data augmentation for real frames
datagen = ImageDataGenerator(
    rotation_range=15,  # Increased rotation range
    width_shift_range=0.1,  # Increased width shift range
    height_shift_range=0.1,  # Increased height shift range
    shear_range=0.1,  # Added shear range
    zoom_range=0.1,  # Added zoom range
    horizontal_flip=True,
    fill_mode='nearest'  # Fill mode to handle new pixels
)

# Augment frames without saving
def augment_frames(image_folder):
    augmented_frames = []
    for filename in os.listdir(image_folder):
        image_path = os.path.join(image_folder, filename)
        image = cv2.imread(image_path)

        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB
            image = cv2.resize(image, (256, 256))  # Resize image to a consistent size
            image = image.reshape((1,) + image.shape)  # Reshape for datagen

            augmented_iter = datagen.flow(image, batch_size=1)
            for _ in range(5):  # Generate 5 augmented images for each original image
                augmented_frames.append(next(augmented_iter)[0].astype(np.uint8))  # Take one augmented frame

    return augmented_frames

# Load and augment real frames only
augmented_real_frames = augment_frames(real_frames_path)
print(f"Augmented {len(augmented_real_frames)} real frames.")

Augmented 12690 real frames.


In [ ]:

# Define minimal data augmentation for real frames
datagen = ImageDataGenerator(
    rotation_range=5,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

# Augment frames without saving
def augment_frames(image_folder):
    augmented_frames = []
    for filename in os.listdir(image_folder):
        image_path = os.path.join(image_folder, filename)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB
        image = image.reshape((1,) + image.shape)  # Reshape for datagen
        augmented_iter = datagen.flow(image, batch_size=1)
        augmented_frames.append(next(augmented_iter)[0].astype(np.uint8))  # Take one augmented frame
    return augmented_frames

# Load and augment real frames only
augmented_real_frames = augment_frames(real_frames_path)
print(f"Augmented {len(augmented_real_frames)} real frames.")

Augmented 2538 real frames.


In [ ]:
# Load fake frames
fake_frames = [cv2.cvtColor(cv2.imread(os.path.join(fake_frames_path, filename)), cv2.COLOR_BGR2RGB) for filename in os.listdir(fake_frames_path)]
print(f"Loaded {len(fake_frames)} fake frames.")

Loaded 12888 fake frames.


In [ ]:
# def load_and_process_fake_frames(fake_frames_path, batch_size=32):
#     """Loads and processes fake frames in batches."""
#     filenames = os.listdir(fake_frames_path)
#     num_batches = (len(filenames) + batch_size - 1) // batch_size

#     for batch_idx in range(num_batches):
#         start_idx = batch_idx * batch_size
#         end_idx = min((batch_idx + 1) * batch_size, len(filenames))
#         batch_filenames = filenames[start_idx:end_idx]

#         batch_frames = []
#         for filename in batch_filenames:
#             image_path = os.path.join(fake_frames_path, filename)
#             image = cv2.imread(image_path)
#             image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#             batch_frames.append(image)

#         yield batch_frames  # Yield the processed batch

# # Initialize a list to store all fake frames
# fake_frames = []
# # Iterate through batches of fake frames
# for batch_frames in load_and_process_fake_frames(fake_frames_path):
#     # Extend the fake_frames list with the current batch
#     fake_frames.extend(batch_frames)

# print(f"Loaded {len(fake_frames)} fake frames.")

In [ ]:
# Resize fake frames before converting to numpy array
fake_frames = [cv2.resize(frame, (256, 256)) for frame in fake_frames]

# Ensure all augmented real frames are the same size
augmented_real_frames = [cv2.resize(frame, (256, 256)) for frame in augmented_real_frames]

# Convert lists to numpy arrays
augmented_real_frames = np.array(augmented_real_frames)
fake_frames = np.array(fake_frames)

In [ ]:
!pip install --upgrade tensorflow
!pip install vit-keras
!pip install --upgrade tensorflow
!pip install tensorflow tensorflow-addons vit-keras


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, LayerNormalization, Add, Input, Flatten, Dense, Activation
from tensorflow.keras.models import Model

# Define a simplified PVT block (remove @tf.function if present)
def pvt_block(x, num_channels):
    shortcut = x
    # Initialize layers outside the function to avoid creating them during tracing
    conv1 = Conv2D(num_channels, kernel_size=1, strides=1, padding='same')
    norm1 = LayerNormalization()
    conv2 = Conv2D(num_channels, kernel_size=3, strides=1, padding='same')
    norm2 = LayerNormalization()
    proj = Conv2D(num_channels * 4, kernel_size=1, strides=1, padding='same')
    conv3 = Conv2D(num_channels * 4, kernel_size=1, strides=1, padding='same')
    norm3 = LayerNormalization()
    add = Add()

    x = conv1(x)  # Use the pre-initialized layers
    x = norm1(x)
    # Use Keras Activation layer for ReLU
    x = Activation('relu')(x)
    x = conv2(x)
    x = norm2(x)
    # Use Keras Activation layer for ReLU
    x = Activation('relu')(x)
    # Project the shortcut to match the number of channels
    shortcut = proj(shortcut)
    x = conv3(x)
    x = norm3(x)
    x = add([shortcut, x])
    # Use Keras Activation layer for ReLU
    x = Activation('relu')(x)
    return x


# model for rgb images in first pipeline

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, LayerNormalization, Add, Input, Flatten, Dense # Import Flatten and Dense
from tensorflow.keras.models import Model

# Define the model
input_shape = (224, 224, 3)  # Adjust this based on your frame size
inputs = Input(shape=input_shape)
x = pvt_block(inputs, num_channels=8)
# Add more PVT blocks or layers as needed

x = Flatten()(x)
binary_output = Dense(1, activation='sigmoid', name='binary_output')(x)
distillation_output = Dense(1, activation='sigmoid', name='distillation_output')(x)
multimodal_output = Dense(1, activation='sigmoid', name='multimodal_output')(x)

modelrgb = Model(inputs, [binary_output, distillation_output, multimodal_output])

# Compile the model with multiple loss functions and metrics
modelrgb.compile(
    optimizer='adam',
    loss={'binary_output': 'binary_crossentropy', 'distillation_output': 'binary_crossentropy', 'multimodal_output': 'binary_crossentropy'},
    metrics={'binary_output': 'accuracy', 'distillation_output': 'accuracy', 'multimodal_output': 'accuracy'}
)

# Display the model summary
modelrgb.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 224, 224, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d (Conv2D)           │ (None, 224, 224, 8)    │             32 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization       │ (None, 224, 224, 8)    │             16 │ conv2d[0][0]           │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation (Activation)   │ (None, 224, 224, 8)    │              0 │ layer_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_1 (Conv2D)         │ (None, 224, 224, 8)    │            584 │ activation[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_1     │ (None, 224, 224, 8)    │             16 │ conv2d_1[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_1 (Activation) │ (None, 224, 224, 8)    │              0 │ layer_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_3 (Conv2D)         │ (None, 224, 224, 32)   │            288 │ activation_1[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_2 (Conv2D)         │ (None, 224, 224, 32)   │            128 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_2     │ (None, 224, 224, 32)   │             64 │ conv2d_3[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add (Add)                 │ (None, 224, 224, 32)   │              0 │ conv2d_2[0][0],        │
│                           │                        │                │ layer_normalization_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_2 (Activation) │ (None, 224, 224, 32)   │              0 │ add[0][0]              │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten (Flatten)         │ (None, 1605632)        │              0 │ activation_2[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ binary_output (Dense)     │ (None, 1)              │      1,605,633 │ flatten[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ distillation_output       │ (None, 1)              │      1,605,633 │ flatten[0][0]          │
│ (Dense)                   │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multimodal_output (Dense) │ (None, 1)              │      1,605,633 │ flatten[0][0]          │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 4,818,027 (18.38 MB)

 Trainable params: 4,818,027 (18.38 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Prepare the dataset
X = np.concatenate((augmented_real_frames, fake_frames))
# Resize all frames to match model input
X = np.array([cv2.resize(frame, (224, 224)) for frame in X])
y = np.concatenate((np.ones(len(augmented_real_frames)), np.zeros(len(fake_frames))))

# Split the dataset into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Convert NumPy arrays to TensorFlow tensors
X_train = tf.convert_to_tensor(X_train)
y_train = tf.convert_to_tensor(y_train)

# Reshape y_train to have a shape of (samples, 1) for binary classification
y_train = tf.reshape(y_train, (-1, 1))

# Train the model
modelrgb.fit(X_train, {'binary_output': y_train, 'distillation_output': y_train, 'multimodal_output': y_train},
             epochs=10, batch_size=16, validation_split=0.2)

Epoch 1/10
1024/1024 ━━━━━━━━━━━━━━━━━━━━ 47s 36ms/step - binary_output_accuracy: 0.7157 - binary_output_loss: 413.6354 - distillation_output_accuracy: 0.7224 - distillation_output_loss: 395.0204 - loss: 1147.0847 - multimodal_output_accuracy: 0.7296 - multimodal_output_loss: 338.4292 - val_binary_output_accuracy: 0.8600 - val_binary_output_loss: 22.1073 - val_distillation_output_accuracy: 0.8976 - val_distillation_output_loss: 11.2503 - val_loss: 46.2172 - val_multimodal_output_accuracy: 0.8849 - val_multimodal_output_loss: 12.8266
Epoch 2/10
1024/1024 ━━━━━━━━━━━━━━━━━━━━ 32s 31ms/step - binary_output_accuracy: 0.9092 - binary_output_loss: 9.2942 - distillation_output_accuracy: 0.9158 - distillation_output_loss: 7.2659 - loss: 23.5929 - multimodal_output_accuracy: 0.9172 - multimodal_output_loss: 7.0327 - val_binary_output_accuracy: 0.8761 - val_binary_output_loss: 11.2035 - val_distillation_output_accuracy: 0.9226 - val_distillation_output_loss: 7.7070 - val_loss: 34.4110 - val_mult

In [ ]:
# Get predictions for training data
predictions_train = modelrgb.predict(X_train)
distillation_predictions_train = predictions_train[1] # Access the second element of the list, which corresponds to distillation output
multimodal_predictions_train = predictions_train[2]   # Access the third element, which corresponds to multimodal output

# Get predictions for testing data
predictions_test = modelrgb.predict(X_test)
distillation_predictions_test = predictions_test[1]
multimodal_predictions_test = predictions_test[2]

640/640 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step
160/160 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step


In [ ]:
# Assuming you have already split your data into X_test and y_test

# Evaluate the model on test data
accuracy = modelrgb.evaluate(X_test, y_test)

# Print the accuracy
print(f"Model accuracy: {accuracy[1] * 100:.2f}%")  # Assuming accuracy is the 5th metric in your model's metrics list

160/160 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - binary_output_accuracy: 0.9314 - binary_output_loss: 0.4528 - distillation_output_accuracy: 0.0000e+00 - distillation_output_loss: 0.0000e+00 - loss: 0.4528 - multimodal_output_accuracy: 0.0000e+00 - multimodal_output_loss: 0.0000e+00
Model accuracy: 48.35%


In [ ]:

# # Evaluate the model on test data
# accuracy = modelrgb.evaluate(X_test, y_test)

# # Print the evaluation results
# print(f"Model evaluation results:")
# print(f"Loss: {accuracy[0]}")
# print(f"Binary Output Accuracy: {accuracy[5] * 100:.2f}%")
# print(f"Distillation Output Accuracy: {accuracy[6] * 100:.2f}%")
# print(f"Multimodal Output Accuracy: {accuracy[6] * 100:.2f}%")

# SECOND PIPELINE

TRYING BYCHANGING THE FUNC COMPLETELY


In [ ]:
import librosa
import numpy as np
from scipy.fftpack import fft
from scipy.signal import butter, lfilter
import cv2
import os

# High-pass filter to remove low-frequency noise
def high_pass_filter(signal, cutoff_freq=100, sample_rate=22050):
    nyquist_freq = 0.5 * sample_rate
    normal_cutoff = cutoff_freq / nyquist_freq
    b, a = butter(4, normal_cutoff, btype='high', analog=False)
    filtered_signal = lfilter(b, a, signal)
    return filtered_signal

# Extract both MFCC and FFT features from an image
def extract_mfcc_fft(image, sample_rate=22050, n_mfcc=13):
    # Convert image to grayscale
    image_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    signal = image_gray.mean(axis=0)  # Flatten the image into a 1D signal by averaging across one axis

    # Apply high-pass filter to clean the signal
    filtered_signal = high_pass_filter(signal)

    # Adjust n_fft based on signal length for FFT
    n_fft = min(2048, len(filtered_signal))

    # Extract MFCC features using librosa (treating the image signal like an audio signal)
    mfccs = librosa.feature.mfcc(y=filtered_signal, sr=sample_rate, n_mfcc=n_mfcc, n_fft=n_fft)

    # Extract FFT features
    fft_features = np.abs(fft(filtered_signal, n=n_fft))

    return mfccs, fft_features

# Example usage
# Extract features from augmented real frames
real_features = [extract_mfcc_fft(frame) for frame in augmented_real_frames]

# Extract features from fake frames (not augmented)
fake_features = []
for filename in os.listdir(fake_frames_path):
    image_path = os.path.join(fake_frames_path, filename)
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mfcc, fft_features = extract_mfcc_fft(image)
    fake_features.append((mfcc, fft_features))

print(f"Extracted features from {len(real_features)} augmented real frames.")
print(f"Extracted features from {len(fake_features)} fake frames.")

/usr/local/lib/python3.10/dist-packages/librosa/feature/spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


Extracted features from 12690 augmented real frames.
Extracted features from 12888 fake frames.


extracting the features of mfcc and fft and feeding in  model lATER

In [ ]:
import librosa
import numpy as np
from scipy.fftpack import fft
from scipy.signal import butter, lfilter

def extract_mfcc_fft(image):
    image_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    signal = image_gray.mean(axis=0)

    # High-pass filter
    cutoff_freq = 100  # Adjust this value based on your desired cutoff frequency
    nyquist_freq = 0.5 * 22050
    normal_cutoff = cutoff_freq / nyquist_freq
    b, a = butter(4, normal_cutoff, btype='high', analog=False)
    filtered_signal = lfilter(b, a, signal)

    # Adjust n_fft based on the length of filtered_signal
    n_fft = min(2048, len(filtered_signal))  # Set n_fft to be smaller of 2048 or signal length
    mfccs = librosa.feature.mfcc(y=filtered_signal, sr=22050, n_mfcc=13, n_fft=n_fft)
    fft_features = np.abs(fft(filtered_signal, n=n_fft))
    return mfccs, fft_features

# Example usage
# real_features = [extract_mfcc_fft(frame) for frame in augmented_real_frames]
# fake_features = [extract_mfcc_fft(frame) for frame in fake_frames]

# Ensure to handle varying signal lengths appropriately in your actual application.


# Extract features from augmented real frames
real_features = [extract_mfcc_fft(frame) for frame in augmented_real_frames]

# Extract features from fake frames (not augmented)
fake_features = []
for filename in os.listdir(fake_frames_path):
    image_path = os.path.join(fake_frames_path, filename)
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mfcc, fft_features = extract_mfcc_fft(image)
    fake_features.append((mfcc, fft_features))

print(f"Extracted features from {len(real_features)} augmented real frames.")
print(f"Extracted features from {len(fake_features)} fake frames.")

Extracted features from 12690 augmented real frames.
Extracted features from 12888 fake frames.


trying once with some modif

In [ ]:
import cv2
import librosa
import numpy as np
from scipy.fftpack import fft
from scipy.signal import butter, lfilter
import os

def extract_mfcc_fft(image, sr=22050, cutoff_freq=100, n_mfcc=13, n_fft=2048):
    """
    Extracts high-pass filtered signal, MFCC, and FFT features from an image.
    Assumes the image is a 2D grayscale image.
    """
    # Convert image to grayscale
    image_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Create a 1D signal by averaging pixel intensities across rows
    signal = image_gray.mean(axis=0)

    # High-pass filter
    nyquist_freq = 0.5 * sr
    normal_cutoff = cutoff_freq / nyquist_freq
    b, a = butter(4, normal_cutoff, btype='high', analog=False)
    filtered_signal = lfilter(b, a, signal)

    # Ensure n_fft does not exceed the signal length to prevent crashes
    n_fft = min(n_fft, len(filtered_signal))

    # Extract MFCC and FFT features
    mfccs = librosa.feature.mfcc(y=filtered_signal, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft)
    fft_features = np.abs(fft(filtered_signal, n=n_fft))

    return mfccs, fft_features

# Example paths and frames - You should replace these with your actual data
augmented_real_frames = []  # Your augmented real frames should be loaded here
fake_frames_path = '/content/drive/MyDrive/Celeb-DF/Frames-Fake'  # Path to your fake frames directory

# Process augmented real frames
real_features = []
for frame in augmented_real_frames:
    mfcc, fft_features = extract_mfcc_fft(frame)
    real_features.append((mfcc, fft_features))

# Process fake frames
fake_features = []
for filename in os.listdir(fake_frames_path):
    image_path = os.path.join(fake_frames_path, filename)
    image = cv2.imread(image_path)
    if image is not None:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mfcc, fft_features = extract_mfcc_fft(image)
        fake_features.append((mfcc, fft_features))

# Output information
print(f"Extracted features from {len(real_features)} augmented real frames.")
print(f"Extracted features from {len(fake_features)} fake frames.")


/usr/local/lib/python3.10/dist-packages/librosa/feature/spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


Extracted features from 0 augmented real frames.
Extracted features from 12888 fake frames.


SOME MODIF IN THE FEATURE CODE PADDING

In [ ]:
import numpy as np

# Assuming you have extracted features stored in real_features and fake_features
# real_features and fake_features should be lists of tuples (mfccs, fft_features)

# Pad or truncate MFCC features to a consistent length
max_length_mfcc = max(len(f[0][0]) for f in real_features + fake_features)  # Maximum time steps across all MFCCs
X_mfcc = []
for f in real_features + fake_features:
    mfcc = f[0]
    if len(mfcc[0]) < max_length_mfcc:
        # Pad with zeros along the time axis (axis 1 for MFCC)
        mfcc = np.pad(mfcc, ((0, 0), (0, max_length_mfcc - len(mfcc[0]))), mode='constant')
    elif len(mfcc[0]) > max_length_mfcc:
        # Truncate along the time axis
        mfcc = mfcc[:, :max_length_mfcc]
    X_mfcc.append(mfcc)

X_mfcc = np.array(X_mfcc)

# Pad or truncate FFT features to a consistent length
max_length_fft = max(len(f[1]) for f in real_features + fake_features)  # Maximum FFT feature length
X_fft = []
for f in real_features + fake_features:
    fft_features = f[1]
    if len(fft_features) < max_length_fft:
        # Pad with zeros
        fft_features = np.pad(fft_features, (0, max_length_fft - len(fft_features)), mode='constant')
    elif len(fft_features) > max_length_fft:
        # Truncate
        fft_features = fft_features[:max_length_fft]
    X_fft.append(fft_features)

X_fft = np.array(X_fft)

# Create labels: 0 for real, 1 for fake
y_labels = np.array([0] * len(real_features) + [1] * len(fake_features))

# Print shapes to confirm the consistency of features and labels
print(f"MFCC features shape: {X_mfcc.shape}")  # Should be (num_samples, n_mfcc, max_length_mfcc)
print(f"FFT features shape: {X_fft.shape}")    # Should be (num_samples, max_length_fft)
print(f"Labels shape: {y_labels.shape}")       # Should be (num_samples,)


MFCC features shape: (25578, 13, 2)
FFT features shape: (25578, 974)
Labels shape: (25578,)


In [ ]:
import numpy as np

# Assuming you have extracted features stored in real_features and fake_features
# real_features and fake_features should be lists of tuples (mfccs, fft_features)

# Pad or truncate MFCC features to a consistent length
max_length_mfcc = max(len(f[0][0]) for f in real_features + fake_features)  # Find the maximum length
X_mfcc = []
for f in real_features + fake_features:
    mfcc = f[0]
    if len(mfcc[0]) < max_length_mfcc:
        # Pad with zeros along the time axis
        mfcc = np.pad(mfcc, ((0, 0), (0, max_length_mfcc - len(mfcc[0]))), mode='constant')
    elif len(mfcc[0]) > max_length_mfcc:
        # Truncate along the time axis
        mfcc = mfcc[:, :max_length_mfcc]
    X_mfcc.append(mfcc)

X_mfcc = np.array(X_mfcc)

# Pad or truncate FFT features to a consistent length
max_length_fft = max(len(f[1]) for f in real_features + fake_features)
X_fft = []
for f in real_features + fake_features:
    fft_features = f[1]
    if len(fft_features) < max_length_fft:
        # Pad with zeros
        fft_features = np.pad(fft_features, (0, max_length_fft - len(fft_features)), mode='constant')
    elif len(fft_features) > max_length_fft:
        # Truncate
        fft_features = fft_features[:max_length_fft]
    X_fft.append(fft_features)

X_fft = np.array(X_fft)

# Create labels: 0 for real, 1 for fake
y_labels = np.array([0] * len(real_features) + [1] * len(fake_features))

print(f"MFCC features shape: {X_mfcc.shape}")
print(f"FFT features shape: {X_fft.shape}")
print(f"Labels shape: {y_labels.shape}")

MFCC features shape: (25578, 13, 2)
FFT features shape: (25578, 974)
Labels shape: (25578,)


In [ ]:
from tensorflow.keras.layers import Input
import numpy as np

# Assuming X_mfcc and X_fft are already defined
input_mfcc = Input(shape=X_mfcc.shape[1:], name='input_mfcc')
input_fft = Input(shape=X_fft.shape[1:], name='input_fft')

ADAPTIVE FUSION TRIED HERE

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal, GlorotUniform
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 512  # Increased projection dimension
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Adaptive Fusion (instead of Concatenation)
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Adaptive fusion using weighted sum of inputs
adaptive_weights_mfcc = Dense(projection_dim, kernel_initializer=GlorotUniform(), activation='softmax')(patch_mfcc)
adaptive_weights_fft = Dense(projection_dim, kernel_initializer=GlorotUniform(), activation='softmax')(patch_fft)
fused_patches = Add()([adaptive_weights_mfcc * patch_mfcc, adaptive_weights_fft * patch_fft])
fused_patches = Reshape((-1, projection_dim))(fused_patches)

# ViT Block with Batch Normalization
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.2):  # Increased dropout rate for regularization
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    attn_output = BatchNormalization()(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = BatchNormalization()(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT Block
vit_output = vit_block(fused_patches)

# PVT Block with Batch Normalization
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels, kernel_initializer=HeNormal())(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels, kernel_initializer=HeNormal())(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4, kernel_initializer=HeNormal())(shortcut)
    x = Dense(num_channels * 4, kernel_initializer=HeNormal())(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT Blocks after ViT Block
num_pvt_blocks = 5
pvt_output = vit_output
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=512)

# Final Output Layer for Vision Transformer Path
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(pvt_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Adaptive Fusion in Second Pipeline
adaptive_weights_distillation = Dense(1, kernel_initializer=GlorotUniform(), activation='softmax')(input_distillation_second)
adaptive_weights_multimodal = Dense(1, kernel_initializer=GlorotUniform(), activation='softmax')(input_multimodal_second)
fused_outputs_second = Add()([adaptive_weights_distillation * input_distillation_second, adaptive_weights_multimodal * input_multimodal_second])

# Progressive Dense Layers with Regularization and Dropout
dense_layer_1 = Dense(128, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-4))(Flatten()(fused_outputs_second))
dropout_layer_1 = Dropout(0.5)(dense_layer_1)  # Increased dropout rate

dense_layer_2 = Dense(64, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-4))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)  # Increased dropout rate

dense_layer_3 = Dense(32, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-4))(dropout_layer_2)
dropout_layer_3 = Dropout(0.5)(dense_layer_3)  # Increased dropout rate

# Final Output Layer
final_output = Dense(1, activation='sigmoid', name='output')(dropout_layer_3)

# Compile Model with AdamW optimizer (learning rate reduced)
optimizer = AdamW(learning_rate=1e-4, weight_decay=1e-5)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)  # Early stopping at patience 3
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Summary
model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_distillation_second │ (None, 1)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_multimodal_second   │ (None, 1)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_84 (Dense)          │ (None, 1)              │              2 │ input_distillation_se… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_85 (Dense)          │ (None, 1)              │              2 │ input_multimodal_seco… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multiply_10 (Multiply)    │ (None, 1)              │              0 │ dense_84[0][0],        │
│                           │                        │                │ input_distillation_se… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multiply_11 (Multiply)    │ (None, 1)              │              0 │ dense_85[0][0],        │
│                           │                        │                │ input_multimodal_seco… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_28 (Add)              │ (None, 1)              │              0 │ multiply_10[0][0],     │
│                           │                        │                │ multiply_11[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_6 (Flatten)       │ (None, 1)              │              0 │ add_28[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_86 (Dense)          │ (None, 128)            │            256 │ flatten_6[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_17 (Dropout)      │ (None, 128)            │              0 │ dense_86[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_87 (Dense)          │ (None, 64)             │          8,256 │ dropout_17[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_18 (Dropout)      │ (None, 64)             │              0 │ dense_87[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_88 (Dense)          │ (None, 32)             │          2,080 │ dropout_18[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_19 (Dropout)      │ (None, 32)             │              0 │ dense_88[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ output (Dense)            │ (None, 1)              │             33 │ dropout_19[0][0]       │
└──────────────────────

 Total params: 10,629 (41.52 KB)

 Trainable params: 10,629 (41.52 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
import numpy as np

# Modified function to convert one-hot labels to binary before feeding to the generator
def convert_labels_to_binary(generator):
    for x, y in generator:
        # Assuming y is of shape (batch_size, 2)
        y_binary = np.argmax(y, axis=-1)  # Convert one-hot to binary (0 or 1)
        yield x, y_binary

# Apply the conversion before feeding to the model.fit
combined_train_gen = convert_labels_to_binary(combined_train_gen)
combined_val_gen = convert_labels_to_binary(combined_val_gen)

# Model training (rest of the code remains the same)
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=15,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

ValueError: When passing a dataset to a Keras model, the arrays must be at least rank 1. Received: 0 of rank 0.

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Flatten, Conv1D, BatchNormalization, GlobalAveragePooling1D, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal, GlorotUniform
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 512  # Increased projection dimension
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape for processing
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Weighted sum (Adaptive Fusion)
adaptive_weights_mfcc = Dense(projection_dim, kernel_initializer=GlorotUniform(), activation='sigmoid')(patch_mfcc)
adaptive_weights_fft = Dense(projection_dim, kernel_initializer=GlorotUniform(), activation='sigmoid')(patch_fft)
fused_patches = Add()([adaptive_weights_mfcc * patch_mfcc, adaptive_weights_fft * patch_fft])

# Product Multiplication for additional fusion
product_fusion = Multiply()([patch_mfcc, patch_fft])  # Product multiplication of the embeddings

# Combine fused patches with product fusion
combined_fused_patches = Add()([fused_patches, product_fusion])  # Combine adaptive fusion and product multiplication

# ViT Block with Batch Normalization
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.2):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    attn_output = BatchNormalization()(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = BatchNormalization()(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT Block
vit_output = vit_block(combined_fused_patches)

# PVT Block with Batch Normalization
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels, kernel_initializer=HeNormal())(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels, kernel_initializer=HeNormal())(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4, kernel_initializer=HeNormal())(shortcut)
    x = Dense(num_channels * 4, kernel_initializer=HeNormal())(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT Blocks after ViT Block
num_pvt_blocks = 5
pvt_output = vit_output
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=512)

# Optionally add Global Average Pooling
pooled_output = GlobalAveragePooling1D()(pvt_output)

# Final Output Layer for Vision Transformer Path
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(pvt_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Adaptive Fusion in Second Pipeline
adaptive_weights_distillation = Dense(1, kernel_initializer=GlorotUniform(), activation='sigmoid')(input_distillation_second)
adaptive_weights_multimodal = Dense(1, kernel_initializer=GlorotUniform(), activation='sigmoid')(input_multimodal_second)
fused_outputs_second = Add()([adaptive_weights_distillation * input_distillation_second, adaptive_weights_multimodal * input_multimodal_second])

# Progressive Dense Layers with Regularization and Dropout
dense_layer_1 = Dense(128, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-4))(Flatten()(fused_outputs_second))
dropout_layer_1 = Dropout(0.5)(dense_layer_1)

dense_layer_2 = Dense(64, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-4))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)

dense_layer_3 = Dense(32, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-4))(dropout_layer_2)
dropout_layer_3 = Dropout(0.5)(dense_layer_3)

# Final Output Layer
final_output = Dense(1, activation='sigmoid', name='output')(dropout_layer_3)

# Compile Model with AdamW optimizer (learning rate reduced)
optimizer = AdamW(learning_rate=1e-4, weight_decay=1e-5)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])


model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_distillation_second │ (None, 1)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_multimodal_second   │ (None, 1)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_233 (Dense)         │ (None, 1)              │              2 │ input_distillation_se… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_234 (Dense)         │ (None, 1)              │              2 │ input_multimodal_seco… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multiply_37 (Multiply)    │ (None, 1)              │              0 │ dense_233[0][0],       │
│                           │                        │                │ input_distillation_se… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multiply_38 (Multiply)    │ (None, 1)              │              0 │ dense_234[0][0],       │
│                           │                        │                │ input_multimodal_seco… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_79 (Add)              │ (None, 1)              │              0 │ multiply_37[0][0],     │
│                           │                        │                │ multiply_38[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_16 (Flatten)      │ (None, 1)              │              0 │ add_79[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_235 (Dense)         │ (None, 128)            │            256 │ flatten_16[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_47 (Dropout)      │ (None, 128)            │              0 │ dense_235[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_236 (Dense)         │ (None, 64)             │          8,256 │ dropout_47[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_48 (Dropout)      │ (None, 64)             │              0 │ dense_236[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_237 (Dense)         │ (None, 32)             │          2,080 │ dropout_48[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_49 (Dropout)      │ (None, 32)             │              0 │ dense_237[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ output (Dense)            │ (None, 1)              │             33 │ dropout_49[0][0]       │
└──────────────────────

 Total params: 10,629 (41.52 KB)

 Trainable params: 10,629 (41.52 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os
def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))

        # Ensure image_labels is one-hot encoded and of type float32
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2).astype(np.float32)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0), dtype=np.float32)  # Ensure matching batch size and data type

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0), dtype=np.float32)  # Ensure matching batch size and data type

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        # Ensure all inputs and targets are float32 tensors
        inputs = [tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs]
        targets = tf.convert_to_tensor(targets, dtype=tf.float32)
        yield (tuple(inputs), targets)

# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=15,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/15


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 2), output.shape=(None, 1)

VIT BLCOK REMOVED HERE

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 256
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Concatenate
patch_fft = Reshape((-1, projection_dim))(patch_fft)
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# PVT Block
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT Blocks
num_pvt_blocks = 5  # Experiment with fewer blocks
pvt_output = combined_patches
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=256)

# No ViT Block

# Final Output Layer
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(pvt_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate Outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Dense Layers
flattened_output = Flatten()(merged_output)
dense_layer_1 = Dense(64, activation='relu', kernel_regularizer=l2(1e-5))(flattened_output)  # Reduced size
dropout_layer_1 = Dropout(0.5)(dense_layer_1)
dense_layer_2 = Dense(32, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)

# Final Output Layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_2)

# Compile Model
optimizer = Adam(learning_rate=1e-4)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape (Reshape)         │ (None, 1, 256)         │              0 │ dense_1[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 14, 256)        │              0 │ dense[0][0],           │
│                           │                        │                │ reshape[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_1 (Reshape)       │ (None, 14, 256)        │              0 │ concatenate[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_2 (Dense)           │ (None, 14, 256)        │         65,792 │ reshape_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 14, 256)        │          1,024 │ dense_2[0][0]          │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu (ReLU)              │ (None, 14, 256)        │              0 │ batch_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_3 (Dense)           │ (None, 14, 256)        │         65,792 │ re_lu[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_1     │ (None, 14, 256)        │          1,024 │ dense_3[0][0]          │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_1 (ReLU)            │ (None, 14, 256)        │              0 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_5 (Dense)           │ (None, 14, 1024)       │        263,168 │ re_lu_1[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_4 (Dense)           │ (None, 14, 1024)       │        263,168 │ reshape_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_3     │ (None, 14, 1024)       │          2,048 │ dense_5[0][0]          │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_1 (Add)               │ (None, 14, 1024)       │              0 │ dense_4[0][0],         │
│                           │                        │                │ layer_normalization_3… │
├──────────────────────

 Total params: 7,523,748 (28.70 MB)

 Trainable params: 7,518,628 (28.68 MB)

 Non-trainable params: 5,120 (20.00 KB)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=5,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.5206 - loss: 0.6967Found 1541 images belonging to 2 classes.


KeyboardInterrupt: 

pvt block removed, vit left

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 256
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Concatenate
patch_fft = Reshape((-1, projection_dim))(patch_fft)
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# ViT Block (kept)
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT Block
vit_output = vit_block(combined_patches)

# Final Output Layer
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(vit_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate Outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Dense Layers
flattened_output = Flatten()(merged_output)
dense_layer_1 = Dense(64, activation='relu', kernel_regularizer=l2(1e-5))(flattened_output)  # Reduced size
dropout_layer_1 = Dropout(0.5)(dense_layer_1)
dense_layer_2 = Dense(32, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)

# Final Output Layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_2)

# Compile Model
optimizer = Adam(learning_rate=1e-4)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_25 (Dense)          │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_24 (Dense)          │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_2 (Reshape)       │ (None, 1, 256)         │              0 │ dense_25[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_2             │ (None, 14, 256)        │              0 │ dense_24[0][0],        │
│ (Concatenate)             │                        │                │ reshape_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_3 (Reshape)       │ (None, 14, 256)        │              0 │ concatenate_2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention      │ (None, 14, 256)        │      2,103,552 │ reshape_3[0][0],       │
│ (MultiHeadAttention)      │                        │                │ reshape_3[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_3 (Dropout)       │ (None, 14, 256)        │              0 │ multi_head_attention[… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_6 (Add)               │ (None, 14, 256)        │              0 │ dropout_3[0][0],       │
│                           │                        │                │ reshape_3[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_8     │ (None, 14, 256)        │            512 │ add_6[0][0]            │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d (Conv1D)           │ (None, 14, 512)        │        393,728 │ layer_normalization_8… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_1 (Conv1D)         │ (None, 14, 512)        │        786,944 │ conv1d[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_2 (Conv1D)         │ (None, 14, 256)        │        131,328 │ conv1d_1[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_4 (Dropout)       │ (None, 14, 256)        │              0 │ conv1d_2[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_3 (Conv1D)         │ (None, 14, 256)        │         65,792 │ layer_normalization_8… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_7 (Add)               │ (None, 14, 256)        │              0 │ dropout_4[0][0],       │
│                           │                        │                │ conv1d_3[0][0]         │
├──────────────────────

 Total params: 3,742,372 (14.28 MB)

 Trainable params: 3,742,372 (14.28 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=5,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step - accuracy: 0.7220 - loss: 0.5985Found 1541 images belonging to 2 classes.
392/392 ━━━━━━━━━━━━━━━━━━━━ 200s 472ms/step - accuracy: 0.7221 - loss: 0.5984 - val_accuracy: 0.8359 - val_loss: 0.4686
Epoch 2/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 74s 164ms/step - accuracy: 0.8147 - loss: 0.5143 - val_accuracy: 0.8363 - val_loss: 0.4532
Epoch 3/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 159ms/step - accuracy: 0.8235 - loss: 0.4927 - val_accuracy: 0.8363 - val_loss: 0.4510
Epoch 4/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 61s 156ms/step - accuracy: 0.8152 - loss: 0.5038 - val_accuracy: 0.8357 - val_loss: 0.4500
Epoch 5/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 59s 150ms/step - accuracy: 0.8196 - loss: 0.4971 - val_accuracy: 0.8449 - val_loss: 0.4379


KEEP PVT AND REMOVE VIT HERE

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, Dropout, Concatenate, Flatten, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 256
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Concatenate
patch_fft = Reshape((-1, projection_dim))(patch_fft)
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# PVT Block
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT Blocks
num_pvt_blocks = 5  # Number of PVT blocks to apply
pvt_output = combined_patches
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=256)

# Flatten the output of PVT Block
flattened_output = Flatten()(pvt_output)

# Final Output Layer for PVT Block
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(flattened_output)

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate Outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Dense Layers
dense_layer_1 = Dense(64, activation='relu', kernel_regularizer=l2(1e-5))(merged_output)
dropout_layer_1 = Dropout(0.5)(dense_layer_1)
dense_layer_2 = Dense(32, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)

# Final Output Layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_2)

# Compile Model
optimizer = Adam(learning_rate=1e-4)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_29 (Dense)          │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_28 (Dense)          │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_4 (Reshape)       │ (None, 1, 256)         │              0 │ dense_29[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_4             │ (None, 14, 256)        │              0 │ dense_28[0][0],        │
│ (Concatenate)             │                        │                │ reshape_4[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_5 (Reshape)       │ (None, 14, 256)        │              0 │ concatenate_4[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_30 (Dense)          │ (None, 14, 256)        │         65,792 │ reshape_5[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_10    │ (None, 14, 256)        │          1,024 │ dense_30[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_15 (ReLU)           │ (None, 14, 256)        │              0 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_31 (Dense)          │ (None, 14, 256)        │         65,792 │ re_lu_15[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_11    │ (None, 14, 256)        │          1,024 │ dense_31[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_16 (ReLU)           │ (None, 14, 256)        │              0 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_33 (Dense)          │ (None, 14, 1024)       │        263,168 │ re_lu_16[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_32 (Dense)          │ (None, 14, 1024)       │        263,168 │ reshape_5[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_10    │ (None, 14, 1024)       │          2,048 │ dense_33[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_8 (Add)               │ (None, 14, 1024)       │              0 │ dense_32[0][0],        │
│                           │                        │                │ layer_normalization_1… │
├──────────────────────

 Total params: 7,523,748 (28.70 MB)

 Trainable params: 7,518,628 (28.68 MB)

 Non-trainable params: 5,120 (20.00 KB)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=5,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.7418 - loss: 0.5792Found 1541 images belonging to 2 classes.
392/392 ━━━━━━━━━━━━━━━━━━━━ 123s 248ms/step - accuracy: 0.7419 - loss: 0.5791 - val_accuracy: 0.8359 - val_loss: 0.4558
Epoch 2/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 81s 170ms/step - accuracy: 0.8173 - loss: 0.5074 - val_accuracy: 0.8383 - val_loss: 0.4489
Epoch 3/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 159ms/step - accuracy: 0.8200 - loss: 0.5047 - val_accuracy: 0.8317 - val_loss: 0.4556
Epoch 4/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 61s 155ms/step - accuracy: 0.8187 - loss: 0.4999 - val_accuracy: 0.8383 - val_loss: 0.4480
Epoch 5/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 60s 154ms/step - accuracy: 0.8212 - loss: 0.4899 - val_accuracy: 0.8370 - val_loss: 0.4484


VIT FIRST AND THEN PVT BLOCK HERE

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 256
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Concatenate
patch_fft = Reshape((-1, projection_dim))(patch_fft)
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# ViT Block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT Block first
vit_output = vit_block(combined_patches)

# PVT Block
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT Blocks after ViT Block
num_pvt_blocks = 5
pvt_output = vit_output
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=256)

# Final Output Layer
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(pvt_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate Outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Dense Layers
dense_layer_1 = Dense(64, activation='relu', kernel_regularizer=l2(1e-5))(merged_output)
dropout_layer_1 = Dropout(0.5)(dense_layer_1)
dense_layer_2 = Dense(32, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)

# Final Output Layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_2)

# Compile Model
optimizer = Adam(learning_rate=1e-4)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape (Reshape)         │ (None, 1, 256)         │              0 │ dense_1[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 14, 256)        │              0 │ dense[0][0],           │
│                           │                        │                │ reshape[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_1 (Reshape)       │ (None, 14, 256)        │              0 │ concatenate[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention      │ (None, 14, 256)        │      2,103,552 │ reshape_1[0][0],       │
│ (MultiHeadAttention)      │                        │                │ reshape_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_1 (Dropout)       │ (None, 14, 256)        │              0 │ multi_head_attention[… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_1 (Add)               │ (None, 14, 256)        │              0 │ dropout_1[0][0],       │
│                           │                        │                │ reshape_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_3     │ (None, 14, 256)        │            512 │ add_1[0][0]            │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d (Conv1D)           │ (None, 14, 512)        │        393,728 │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_1 (Conv1D)         │ (None, 14, 512)        │        786,944 │ conv1d[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_2 (Conv1D)         │ (None, 14, 256)        │        131,328 │ conv1d_1[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_2 (Dropout)       │ (None, 14, 256)        │              0 │ conv1d_2[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_3 (Conv1D)         │ (None, 14, 256)        │         65,792 │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_2 (Add)               │ (None, 14, 256)        │              0 │ dropout_2[0][0],       │
│                           │                        │                │ conv1d_3[0][0]         │
├──────────────────────

 Total params: 11,006,116 (41.99 MB)

 Trainable params: 11,000,996 (41.97 MB)

 Non-trainable params: 5,120 (20.00 KB)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=10,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/10
 91/392 ━━━━━━━━━━━━━━━━━━━━ 1:44:13 21s/step - accuracy: 0.3583 - loss: 0.7999

KeyboardInterrupt: 

A LOT OF CHANGES TRIED AND TESTED HERE LIKE ......
Progressive Reduction of Dense Layers: Reduced by 2 units at each step for better feature refinement.
Learning Rate Adjustment: Changed to 1e-3 for faster convergence but added ReduceLROnPlateau to dynamically adjust it.
Optimizer: Using AdamW, which applies weight decay, helping regularize large models.
Weight Initialization: Used HeNormal for all Dense layers.
Batch Normalization in ViT Block: Added batch normalization after Dense layers and Conv1D layers for better training stability.
Increased Projection Dimension: Increased from 256 to 512 to give the model more representational capacity.

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 512  # Increased projection dimension
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Concatenate
patch_fft = Reshape((-1, projection_dim))(patch_fft)
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# ViT Block with Batch Normalization
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    attn_output = BatchNormalization()(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = BatchNormalization()(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT Block
vit_output = vit_block(combined_patches)

# PVT Block with Batch Normalization
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT Blocks after ViT Block
num_pvt_blocks = 5
pvt_output = vit_output
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=512)

# Final Output Layer for Vision Transformer Path
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(pvt_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate Outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Progressive Dense Layers with Regularization and Dropout
dense_layer_1 = Dense(128, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(Flatten()(merged_output))
dropout_layer_1 = Dropout(0.5)(dense_layer_1)

dense_layer_2 = Dense(64, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)

dense_layer_3 = Dense(32, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_2)
dropout_layer_3 = Dropout(0.5)(dense_layer_3)

dense_layer_4 = Dense(16, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_3)
dropout_layer_4 = Dropout(0.5)(dense_layer_4)

dense_layer_5 = Dense(8, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_4)
dropout_layer_5 = Dropout(0.5)(dense_layer_5)

dense_layer_6 = Dense(4, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_5)
dropout_layer_6 = Dropout(0.5)(dense_layer_6)

dense_layer_7 = Dense(2, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_6)
dropout_layer_7 = Dropout(0.5)(dense_layer_7)

# Final Output Layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_7)

# Compile Model with AdamW optimizer (weight decay)
optimizer = AdamW(learning_rate=1e-3, weight_decay=1e-5)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Summary
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 512)            │        499,200 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 13, 512)        │          1,536 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape (Reshape)         │ (None, 1, 512)         │              0 │ dense_1[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 14, 512)        │              0 │ dense[0][0],           │
│                           │                        │                │ reshape[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_1 (Reshape)       │ (None, 14, 512)        │              0 │ concatenate[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention      │ (None, 14, 512)        │      8,401,408 │ reshape_1[0][0],       │
│ (MultiHeadAttention)      │                        │                │ reshape_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_1 (Dropout)       │ (None, 14, 512)        │              0 │ multi_head_attention[… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 14, 512)        │          2,048 │ dropout_1[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_1 (Add)               │ (None, 14, 512)        │              0 │ batch_normalization[0… │
│                           │                        │                │ reshape_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_3     │ (None, 14, 512)        │          1,024 │ add_1[0][0]            │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d (Conv1D)           │ (None, 14, 512)        │        786,944 │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_1 (Conv1D)         │ (None, 14, 512)        │        786,944 │ conv1d[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_1     │ (None, 14, 512)        │          2,048 │ conv1d_1[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_2 (Conv1D)         │ (None, 14, 512)        │        262,656 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_2 (Dropout)  

 Total params: 39,978,926 (152.51 MB)

 Trainable params: 39,966,638 (152.46 MB)

 Non-trainable params: 12,288 (48.00 KB)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=10,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/10
256/392 ━━━━━━━━━━━━━━━━━━━━ 48:07 21s/step - accuracy: 0.7773 - loss: 0.6873

here i got 85% accuracy after making so many changes

A LOT OF CHANGES TRIED AND TESTED HERE LIKE ...... Progressive Reduction of Dense Layers: Reduced by 2 units at each step for better feature refinement. Learning Rate Adjustment: Changed to 1e-3 for faster convergence but added ReduceLROnPlateau to dynamically adjust it. Optimizer: Using AdamW, which applies weight decay, helping regularize large models. Weight Initialization: Used HeNormal for all Dense layers. Batch Normalization in ViT Block: Added batch normalization after Dense layers and Conv1D layers for better training stability. Increased Projection Dimension: Increased from 256 to 512 to give the model more representational capacity.




In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 512  # Increased projection dimension
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Concatenate
patch_fft = Reshape((-1, projection_dim))(patch_fft)
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# ViT Block with Batch Normalization
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    attn_output = BatchNormalization()(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = BatchNormalization()(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT Block
vit_output = vit_block(combined_patches)

# PVT Block with Batch Normalization
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT Blocks after ViT Block
num_pvt_blocks = 5
pvt_output = vit_output
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=512)

# Final Output Layer for Vision Transformer Path
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(pvt_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate Outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Progressive Dense Layers with Regularization and Dropout
dense_layer_1 = Dense(128, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(Flatten()(merged_output))
dropout_layer_1 = Dropout(0.4)(dense_layer_1)  # Reduced dropout rate

dense_layer_2 = Dense(64, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(0.4)(dense_layer_2)  # Reduced dropout rate

dense_layer_3 = Dense(32, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_2)
dropout_layer_3 = Dropout(0.4)(dense_layer_3)  # Reduced dropout rate

dense_layer_4 = Dense(16, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_3)
dropout_layer_4 = Dropout(0.4)(dense_layer_4)  # Reduced dropout rate

dense_layer_5 = Dense(8, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_4)
dropout_layer_5 = Dropout(0.4)(dense_layer_5)  # Reduced dropout rate

# Final Output Layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_5)

# Compile Model with AdamW optimizer (weight decay)
optimizer = AdamW(learning_rate=1e-3, weight_decay=1e-5)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Summary
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_7 (Cast)             │ (None, 974)            │              0 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_6 (Cast)             │ (None, 13, 2)          │              0 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_41 (Dense)          │ (None, 512)            │        499,200 │ cast_7[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_40 (Dense)          │ (None, 13, 512)        │          1,536 │ cast_6[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_4 (Reshape)       │ (None, 1, 512)         │              0 │ dense_41[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_4             │ (None, 14, 512)        │              0 │ dense_40[0][0],        │
│ (Concatenate)             │                        │                │ reshape_4[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_5 (Reshape)       │ (None, 14, 512)        │              0 │ concatenate_4[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention_2    │ (None, 14, 512)        │      8,401,408 │ reshape_5[0][0],       │
│ (MultiHeadAttention)      │                        │                │ reshape_5[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_19 (Dropout)      │ (None, 14, 512)        │              0 │ multi_head_attention_… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_12    │ (None, 14, 512)        │          2,048 │ dropout_19[0][0]       │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_11 (Add)              │ (None, 14, 512)        │              0 │ batch_normalization_1… │
│                           │                        │                │ reshape_5[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_15    │ (None, 14, 512)        │          1,024 │ add_11[0][0]           │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_8 (Conv1D)         │ (None, 14, 512)        │        786,944 │ layer_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_9 (Conv1D)         │ (None, 14, 512)        │        786,944 │ conv1d_8[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_13    │ (None, 14, 512)        │          2,048 │ conv1d_9[0][0]         │
│ (BatchNormalization) 

 Total params: 39,978,892 (152.51 MB)

 Trainable params: 39,966,604 (152.46 MB)

 Non-trainable params: 12,288 (48.00 KB)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


thissssssssssss

In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=30,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Epoch 1/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 157ms/step - accuracy: 0.8234 - loss: 0.4698 - val_accuracy: 0.8376 - val_loss: 0.4482
Epoch 2/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 61s 156ms/step - accuracy: 0.8225 - loss: 0.4710 - val_accuracy: 0.8357 - val_loss: 0.4516
Epoch 3/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 158ms/step - accuracy: 0.8238 - loss: 0.4686 - val_accuracy: 0.8366 - val_loss: 0.4494
Epoch 4/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 61s 155ms/step - accuracy: 0.8196 - loss: 0.4752 - val_accuracy: 0.8376 - val_loss: 0.4477
Epoch 5/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 59s 152ms/step - accuracy: 0.8201 - loss: 0.4742 - val_accuracy: 0.8350 - val_loss: 0.4519
Epoch 6/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 57s 145ms/step - accuracy: 0.8166 - loss: 0.4794 - val_accuracy: 0.8290 - val_loss: 0.4605
Epoch 7/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 57s 146ms/step - accuracy: 0.8262 - loss: 0.4648 - val_accuracy: 0.8370 - val_loss: 0.4482
Epoch 8/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 57s 146ms/step - accuracy: 0.8158 - loss: 0

trying if it improves further from 85%

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 512  # Increased projection dimension
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Concatenate
patch_fft = Reshape((-1, projection_dim))(patch_fft)
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# ViT Block with Batch Normalization
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.2):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    attn_output = BatchNormalization()(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = BatchNormalization()(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT Block
vit_output = vit_block(combined_patches)

# PVT Block with Batch Normalization
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT Blocks after ViT Block
num_pvt_blocks = 5
pvt_output = vit_output
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=512)

# Final Output Layer for Vision Transformer Path
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(pvt_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate Outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Progressive Dense Layers with Regularization and Dropout
dropout_rate = 0.5  # Increased dropout rate
dense_layer_1 = Dense(128, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(Flatten()(merged_output))
dropout_layer_1 = Dropout(dropout_rate)(dense_layer_1)

dense_layer_2 = Dense(64, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(dropout_rate)(dense_layer_2)

dense_layer_3 = Dense(32, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_2)
dropout_layer_3 = Dropout(dropout_rate)(dense_layer_3)

dense_layer_4 = Dense(16, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_3)
dropout_layer_4 = Dropout(dropout_rate)(dense_layer_4)

dense_layer_5 = Dense(8, activation='relu', kernel_initializer=HeNormal(), kernel_regularizer=l2(1e-5))(dropout_layer_4)
dropout_layer_5 = Dropout(dropout_rate)(dense_layer_5)

# Final Output Layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_5)

# Compile Model with AdamW optimizer (weight decay)
optimizer = AdamW(learning_rate=1e-4, weight_decay=1e-5)  # Reduced learning rate
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Summary
model.summary()


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_166 (Dense)         │ (None, 512)            │        499,200 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_165 (Dense)         │ (None, 13, 512)        │          1,536 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_12 (Reshape)      │ (None, 1, 512)         │              0 │ dense_166[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_12            │ (None, 14, 512)        │              0 │ dense_165[0][0],       │
│ (Concatenate)             │                        │                │ reshape_12[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_13 (Reshape)      │ (None, 14, 512)        │              0 │ concatenate_12[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention_6    │ (None, 14, 512)        │      8,401,408 │ reshape_13[0][0],      │
│ (MultiHeadAttention)      │                        │                │ reshape_13[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_46 (Dropout)      │ (None, 14, 512)        │              0 │ multi_head_attention_… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_71    │ (None, 14, 512)        │          2,048 │ dropout_46[0][0]       │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_44 (Add)              │ (None, 14, 512)        │              0 │ batch_normalization_7… │
│                           │                        │                │ reshape_13[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_46    │ (None, 14, 512)        │          1,024 │ add_44[0][0]           │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_20 (Conv1D)        │ (None, 14, 512)        │        786,944 │ layer_normalization_4… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_21 (Conv1D)        │ (None, 14, 512)        │        786,944 │ conv1d_20[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_72    │ (None, 14, 512)        │          2,048 │ conv1d_21[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_22 (Conv1D)        │ (None, 14, 512)        │        262,656 │ batch_normalization_7… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_47 (Dropout) 

 Total params: 39,978,892 (152.51 MB)

 Trainable params: 39,966,604 (152.46 MB)

 Non-trainable params: 12,288 (48.00 KB)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=15,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.5724 - loss: 1.7726Found 1541 images belonging to 2 classes.
392/392 ━━━━━━━━━━━━━━━━━━━━ 134s 240ms/step - accuracy: 0.5725 - loss: 1.7718 - val_accuracy: 0.8353 - val_loss: 0.6188
Epoch 2/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 90s 172ms/step - accuracy: 0.6599 - loss: 0.9898 - val_accuracy: 0.8376 - val_loss: 0.6170
Epoch 3/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 65s 165ms/step - accuracy: 0.7105 - loss: 0.7844 - val_accuracy: 0.8330 - val_loss: 0.6013
Epoch 4/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 64s 163ms/step - accuracy: 0.7491 - loss: 0.6852 - val_accuracy: 0.8363 - val_loss: 0.5857
Epoch 5/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 159ms/step - accuracy: 0.7713 - loss: 0.6489 - val_accuracy: 0.8370 - val_loss: 0.5713
Epoch 6/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 59s 150ms/step - accuracy: 0.7855 - loss: 0.6004 - val_accuracy: 0.8376 - val_loss: 0.5586
Epoch 7/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 59s 152ms/st

POSITIONAL ENCODING TRIES HERE IN TRANAFORMERS LAYESR

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, Add, Reshape, Concatenate, Flatten, Conv1D
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.optimizers import Adam
import numpy as np

# Positional Encoding Function
def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    depths = np.arange(depth)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (depths // 2)) / np.float32(depth))
    angle_rads = positions * angle_rates
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return tf.cast(angle_rads, dtype=tf.float32)

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)
projection_dim = 256

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Concatenate
patch_fft = Reshape((-1, projection_dim))(patch_fft)
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# Positional Encoding
length_of_combined_patches = combined_patches.shape[1]
pos_encoding = positional_encoding(length_of_combined_patches, projection_dim)
combined_patches += pos_encoding  # Add positional encoding to the combined patches

# PVT Block (Modified with Self-Attention)
def pvt_block(x, num_channels, num_heads=4, ff_dim=512, dropout_rate=0.1):
    # Multi-Head Attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=num_channels)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    # Feed-Forward Network
    ffn = Dense(ff_dim, activation='relu')(out1)
    ffn = Dense(num_channels)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply PVT Blocks (Using Multi-Head Attention)
num_pvt_blocks = 10
pvt_output = combined_patches
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=projection_dim)

# ViT Block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT Block
vit_output = vit_block(pvt_output)

# Final Output Layer
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(vit_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate Outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Dense Layers
flattened_output = Flatten()(merged_output)
dense_layer_1 = Dense(64, activation='relu')(flattened_output)
dropout_layer_1 = Dropout(0.5)(dense_layer_1)
dense_layer_2 = Dense(32, activation='relu')(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)

# Final Output Layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_2)

# Compile Model
optimizer = Adam(learning_rate=1e-4)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_89 (Dense)          │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_88 (Dense)          │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_10 (Reshape)      │ (None, 1, 256)         │              0 │ dense_89[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_10            │ (None, 14, 256)        │              0 │ dense_88[0][0],        │
│ (Concatenate)             │                        │                │ reshape_10[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_11 (Reshape)      │ (None, 14, 256)        │              0 │ concatenate_10[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_31 (Add)              │ (None, 14, 256)        │              0 │ reshape_11[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention_7    │ (None, 14, 256)        │      1,051,904 │ add_31[0][0],          │
│ (MultiHeadAttention)      │                        │                │ add_31[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_32 (Dropout)      │ (None, 14, 256)        │              0 │ multi_head_attention_… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_32 (Add)              │ (None, 14, 256)        │              0 │ dropout_32[0][0],      │
│                           │                        │                │ add_31[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_32    │ (None, 14, 256)        │            512 │ add_32[0][0]           │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_90 (Dense)          │ (None, 14, 512)        │        131,584 │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_91 (Dense)          │ (None, 14, 256)        │        131,328 │ dense_90[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_33 (Dropout)      │ (None, 14, 256)        │              0 │ dense_91[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_33 (Add)              │ (None, 14, 256)        │              0 │ dropout_33[0][0],      │
│                           │                        │                │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_33    │ (None, 14, 256)        │            512 │ add_33[0][0]           │
│ (LayerNormalization) 

 Total params: 16,900,772 (64.47 MB)

 Trainable params: 16,900,772 (64.47 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=5,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 411ms/step - accuracy: 0.4983 - loss: 0.7151Found 1541 images belonging to 2 classes.
392/392 ━━━━━━━━━━━━━━━━━━━━ 252s 468ms/step - accuracy: 0.4987 - loss: 0.7149 - val_accuracy: 0.8353 - val_loss: 0.4984
Epoch 2/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 109s 171ms/step - accuracy: 0.8168 - loss: 0.5146 - val_accuracy: 0.8357 - val_loss: 0.4520
Epoch 3/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 63s 160ms/step - accuracy: 0.8181 - loss: 0.4991 - val_accuracy: 0.8390 - val_loss: 0.4450
Epoch 4/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 159ms/step - accuracy: 0.8217 - loss: 0.4923 - val_accuracy: 0.8323 - val_loss: 0.4546
Epoch 5/5
392/392 ━━━━━━━━━━━━━━━━━━━━ 61s 157ms/step - accuracy: 0.8127 - loss: 0.5026 - val_accuracy: 0.8376 - val_loss: 0.4461


In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# Define input shapes
input_mfcc_shape = (13, 2)
input_fft_shape = (974,)

# Inputs
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding
projection_dim = 256
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape and Concatenate
patch_fft = Reshape((-1, projection_dim))(patch_fft)
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# PVT Block
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels)(x)
    x = BatchNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT Blocks
num_pvt_blocks = 5  # Experiment with fewer blocks
pvt_output = combined_patches
for _ in range(num_pvt_blocks):
    pvt_output = pvt_block(pvt_output, num_channels=256)

# ViT Block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT Block
vit_output = vit_block(pvt_output)

# Final Output Layer
vision_transformer_output = Dense(2, activation='softmax', name='vision_transformer_output')(Flatten()(vit_output))

# Second Pipeline Inputs
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate Outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Dense Layers
flattened_output = Flatten()(merged_output)
dense_layer_1 = Dense(64, activation='relu', kernel_regularizer=l2(1e-5))(flattened_output)  # Reduced size
dropout_layer_1 = Dropout(0.5)(dense_layer_1)
dense_layer_2 = Dense(32, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)

# Final Output Layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_2)

# Compile Model
optimizer = Adam(learning_rate=1e-4)
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_146 (Dense)         │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_145 (Dense)         │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_16 (Reshape)      │ (None, 1, 256)         │              0 │ dense_146[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_15            │ (None, 14, 256)        │              0 │ dense_145[0][0],       │
│ (Concatenate)             │                        │                │ reshape_16[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_17 (Reshape)      │ (None, 14, 256)        │              0 │ concatenate_15[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_147 (Dense)         │ (None, 14, 256)        │         65,792 │ reshape_17[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 14, 256)        │          1,024 │ dense_147[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_60 (ReLU)           │ (None, 14, 256)        │              0 │ batch_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_148 (Dense)         │ (None, 14, 256)        │         65,792 │ re_lu_60[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_1     │ (None, 14, 256)        │          1,024 │ dense_148[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_61 (ReLU)           │ (None, 14, 256)        │              0 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_150 (Dense)         │ (None, 14, 1024)       │        263,168 │ re_lu_61[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_149 (Dense)         │ (None, 14, 1024)       │        263,168 │ reshape_17[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_77    │ (None, 14, 1024)       │          2,048 │ dense_150[0][0]        │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_35 (Add)              │ (None, 14, 1024)       │              0 │ dense_149[0][0],       │
│                           │                        │                │ layer_normalization_7… │
├──────────────────────

 Total params: 18,654,628 (71.16 MB)

 Trainable params: 18,649,508 (71.14 MB)

 Non-trainable params: 5,120 (20.00 KB)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=15,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Epoch 1/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 65s 167ms/step - accuracy: 0.8195 - loss: 0.4973 - val_accuracy: 0.8376 - val_loss: 0.4483
Epoch 2/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 65s 166ms/step - accuracy: 0.8158 - loss: 0.4966 - val_accuracy: 0.8376 - val_loss: 0.4459
Epoch 3/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 65s 167ms/step - accuracy: 0.8179 - loss: 0.4934 - val_accuracy: 0.8370 - val_loss: 0.4478
Epoch 4/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 64s 164ms/step - accuracy: 0.8188 - loss: 0.4894 - val_accuracy: 0.8250 - val_loss: 0.4652
Epoch 5/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 64s 162ms/step - accuracy: 0.8171 - loss: 0.4873 - val_accuracy: 0.8416 - val_loss: 0.4416
Epoch 6/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 59s 150ms/step - accuracy: 0.8224 - loss: 0.4832 - val_accuracy: 0.8310 - val_loss: 0.4573
Epoch 7/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 60s 152ms/step - accuracy: 0.8195 - loss: 0.4864 - val_accuracy: 0.8429 - val_loss: 0.4396
Epoch 8/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 60s 153ms/step - accuracy: 0.8178 - loss: 0

PVT VIT BLOCKS HEREE

In [ ]:
import os
import cv2
import numpy as np
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Conv1D
from tensorflow.keras.models import Model

# Define input shapes for MFCC and FFT features
input_mfcc_shape = (13, 2)  # MFCC shape: 13 coefficients, 2 time steps (assuming 2 channels)
input_fft_shape = (974,)  # FFT shape: 974 features

# Inputs for MFCC and FFT features
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

flattened_mfcc = Flatten()(input_mfcc)
flattened_fft = Flatten()(input_fft)

# Patch Embedding layer for MFCC and FFT
projection_dim = 256  # Adjust as needed
patch_mfcc = Dense(projection_dim)(input_mfcc)
patch_fft = Dense(projection_dim)(input_fft)


 #from here
# Reshape patch_fft to match the rank of patch_mfcc
patch_fft = Reshape((-1, projection_dim))(patch_fft) # Reshape patch_fft to be 3D

# Replace tf.concat with Keras Concatenate layer
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])

# Reshape for PVT block as before
combined_patches = Reshape((-1, projection_dim))(combined_patches)

#to here

#new added from here

# Reshape patch_fft to match the rank of patch_mfcc
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Concatenate patches using Keras Concatenate layer
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])

# Reshape for PVT block
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# Flatten combined_patches before passing to concatenate_1
flattened_combined_patches = Flatten()(combined_patches)

input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Now pass flattened_combined_patches to concatenate_1
concat_layer = Concatenate()([input_distillation_second,
                               input_multimodal_second,
                               flattened_combined_patches])

#to here

# Pyramid Vision Transformer (PVT) block
def pvt_block(x, num_channels):
    shortcut = x
    # Projection layers
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    # Project the shortcut to match the number of channels
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    return x

# Apply PVT block
pvt_output = pvt_block(combined_patches, num_channels=256)  # Adjust num_channels as needed

# Vision Transformer (ViT) block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    # Multi-head self-attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    # Feed Forward Network
    ffn = Conv1D(filters=ff_dim, kernel_size=1, activation='relu')(out1)
    # Project back to the original dimension
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    # Adjust the number of channels in the skip connection to match FFN output
    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)

    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

vit_output = vit_block(pvt_output)
vision_transformer_output = Dense(1, activation='sigmoid', name='vision_transformer_output')(tf.keras.layers.Flatten()(vit_output))

# Define the inputs for the second pipeline
#multimodal_input = Input(shape=multimodal_output.shape[1:], name='multimodal_input')
#distillation_input = Input(shape=distillation_output.shape[1:], name='distillation_input')

input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

#input_distillation_second = Input(shape=(1,), name='input_distillation_second')
#input_multimodal_second = Input(shape=(1,), name='input_multimodal_second')


# Print shapes of inputs before concatenation
print("Shape of input_distillation_second:", input_distillation_second.shape)
print("Shape of input_multimodal_second:", input_multimodal_second.shape)
print("Shape of vision_transformer_output:", vision_transformer_output.shape)


#added new
# Flatten the vision transformer output before concatenation
flattened_vision_output = Flatten()(vision_transformer_output)

# Concatenate the flattened outputs
merged_output = Concatenate()([input_distillation_second,
                               input_multimodal_second,
                               flattened_vision_output])
# Concatenate outputs
#merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Flatten the merged output before passing it to the Dense layer
flattened_output = tf.keras.layers.Flatten()(merged_output)

# Final classification head
final_output = Dense(2, activation='softmax', name='output')(flattened_output)

# print("Shape of input_mfcc:", tf.shape(input_mfcc))
# print("Shape of input_fft:", tf.shape(input_fft))
# print("Shape of input_distillation_second:", tf.shape(input_distillation_second))
# print("Shape of input_multimodal_second:", tf.shape(input_multimodal_second))

# Create model
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)

# Recompile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Display model summary
model.summary()

Shape of input_distillation_second: (None, 1)
Shape of input_multimodal_second: (None, 1)
Shape of vision_transformer_output: (None, 1)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape (Reshape)         │ (None, 1, 256)         │              0 │ dense_1[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_2 (Reshape)       │ (None, 1, 256)         │              0 │ reshape[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_1             │ (None, 14, 256)        │              0 │ dense[0][0],           │
│ (Concatenate)             │                        │                │ reshape_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_3 (Reshape)       │ (None, 14, 256)        │              0 │ concatenate_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_2 (Dense)           │ (None, 14, 256)        │         65,792 │ reshape_3[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_3     │ (None, 14, 256)        │            512 │ dense_2[0][0]          │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu (ReLU)              │ (None, 14, 256)        │              0 │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_3 (Dense)           │ (None, 14, 256)        │         65,792 │ re_lu[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_4     │ (None, 14, 256)        │            512 │ dense_3[0][0]          │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_1 (ReLU)            │ (None, 14, 256)        │              0 │ layer_normalization_4… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_5 (Dense)           │ (None, 14, 1024)       │        263,168 │ re_lu_1[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_4 (Dense)           │ (None, 14, 1024)       │        263,168 │ reshape_3[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_5     │ (None, 14, 1024)       │          2,048 │ dense_5[0][0]          │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_1 (Add)          

 Total params: 10,231,817 (39.03 MB)

 Trainable params: 10,231,817 (39.03 MB)

 Non-trainable params: 0 (0.00 B)

# MADE CHANGES IN MODEL BELOW TO IMPROVE ACCURACY

In [ ]:
#########NEWWWWWWWWWWWWWWWWWWWW


import os
import cv2
import numpy as np
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Conv1D
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal

# Define input shapes for MFCC and FFT features
input_mfcc_shape = (13, 2)  # MFCC shape: 13 coefficients, 2 time steps (assuming 2 channels)
input_fft_shape = (974,)  # FFT shape: 974 features

# Inputs for MFCC and FFT features
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

flattened_mfcc = Flatten()(input_mfcc)
flattened_fft = Flatten()(input_fft)

# Patch Embedding layer for MFCC and FFT
projection_dim = 256  # Adjust as needed

##OLDDDDD
# patch_mfcc = Dense(projection_dim)(input_mfcc)
# patch_fft = Dense(projection_dim)(input_fft)

####NEWW
# Example: Using HeNormal initializer for Dense layers
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)



 #from here
# Reshape patch_fft to match the rank of patch_mfcc
patch_fft = Reshape((-1, projection_dim))(patch_fft) # Reshape patch_fft to be 3D

# Replace tf.concat with Keras Concatenate layer
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])

# Reshape for PVT block as before
combined_patches = Reshape((-1, projection_dim))(combined_patches)

#to here

#new added from here

# Reshape patch_fft to match the rank of patch_mfcc
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Concatenate patches using Keras Concatenate layer
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])

# Reshape for PVT block
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# Flatten combined_patches before passing to concatenate_1
flattened_combined_patches = Flatten()(combined_patches)

input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Now pass flattened_combined_patches to concatenate_1
concat_layer = Concatenate()([input_distillation_second,
                               input_multimodal_second,
                               flattened_combined_patches])

#to here

# Pyramid Vision Transformer (PVT) block
def pvt_block(x, num_channels):
    shortcut = x
    # Projection layers
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    # Project the shortcut to match the number of channels
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    return x

# Apply PVT block
pvt_output = pvt_block(combined_patches, num_channels=256)  # Adjust num_channels as needed


##new changeeee
# Vision Transformer (ViT) block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    # Multi-head self-attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    # Feed Forward Network
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)  # Adjust kernel_size
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)  # Additional Conv1D layer
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2


vit_output = vit_block(pvt_output)
vision_transformer_output = Dense(1, activation='sigmoid', name='vision_transformer_output')(tf.keras.layers.Flatten()(vit_output))

# Define the inputs for the second pipeline
#multimodal_input = Input(shape=multimodal_output.shape[1:], name='multimodal_input')
#distillation_input = Input(shape=distillation_output.shape[1:], name='distillation_input')

input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

#input_distillation_second = Input(shape=(1,), name='input_distillation_second')
#input_multimodal_second = Input(shape=(1,), name='input_multimodal_second')


# Print shapes of inputs before concatenation
print("Shape of input_distillation_second:", input_distillation_second.shape)
print("Shape of input_multimodal_second:", input_multimodal_second.shape)
print("Shape of vision_transformer_output:", vision_transformer_output.shape)


#added new
# Flatten the vision transformer output before concatenation
flattened_vision_output = Flatten()(vision_transformer_output)

# Concatenate the flattened outputs
merged_output = Concatenate()([input_distillation_second,
                               input_multimodal_second,
                               flattened_vision_output])
# Concatenate outputs
#merged_output = Concatenate()([input_distillation_second, input_multimodal_second, vision_transformer_output])

# Flatten the merged output before passing it to the Dense layer
flattened_output = tf.keras.layers.Flatten()(merged_output)

##NEWW
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Dropout

# Example: Adding L2 regularization and Dropout
dense_layer = Dense(512, activation='relu', kernel_regularizer=l2(1e-5))(flattened_output)
dropout_layer = Dropout(0.5)(dense_layer)  # Change dropout rate as needed
final_output = Dense(2, activation='softmax', name='output')(dropout_layer)


# print("Shape of input_mfcc:", tf.shape(input_mfcc))
# print("Shape of input_fft:", tf.shape(input_fft))
# print("Shape of input_distillation_second:", tf.shape(input_distillation_second))
# print("Shape of input_multimodal_second:", tf.shape(input_multimodal_second))

# Define optimizer with custom learning rate
from tensorflow.keras.optimizers import Adam

# Original optimizer (for reference)
# optimizer = 'adam'

# Example: Changing to Adam with a custom learning rate
optimizer = Adam(learning_rate=1e-4)  # Change learning rate as needed

# Create model
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)

# Compile the model with the new optimizer
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Shape of input_distillation_second: (None, 1)
Shape of input_multimodal_second: (None, 1)
Shape of vision_transformer_output: (None, 1)


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_32 (Dense)          │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_12 (Reshape)      │ (None, 1, 256)         │              0 │ dense_32[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_31 (Dense)          │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_14 (Reshape)      │ (None, 1, 256)         │              0 │ reshape_12[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_15            │ (None, 14, 256)        │              0 │ dense_31[0][0],        │
│ (Concatenate)             │                        │                │ reshape_14[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_15 (Reshape)      │ (None, 14, 256)        │              0 │ concatenate_15[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_33 (Dense)          │ (None, 14, 256)        │         65,792 │ reshape_15[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_23    │ (None, 14, 256)        │            512 │ dense_33[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_12 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_34 (Dense)          │ (None, 14, 256)        │         65,792 │ re_lu_12[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_24    │ (None, 14, 256)        │            512 │ dense_34[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_13 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_36 (Dense)          │ (None, 14, 1024)       │        263,168 │ re_lu_13[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_35 (Dense)          │ (None, 14, 1024)       │        263,168 │ reshape_15[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_25    │ (None, 14, 1024)       │          2,048 │ dense_36[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_13 (Add)         

 Total params: 12,070,403 (46.04 MB)

 Trainable params: 12,070,403 (46.04 MB)

 Non-trainable params: 0 (0.00 B)

#channels and number of pvt blocks are changed here

In [ ]:
import os
import cv2
import numpy as np
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Conv1D, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# Define input shapes for MFCC and FFT features
input_mfcc_shape = (13, 2)  # MFCC shape: 13 coefficients, 2 time steps
input_fft_shape = (974,)  # FFT shape: 974 features

# Inputs for MFCC and FFT features
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Flatten MFCC and FFT inputs
flattened_mfcc = Flatten()(input_mfcc)
flattened_fft = Flatten()(input_fft)

# Patch Embedding layer for MFCC and FFT
projection_dim = 256  # Adjust as needed
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(flattened_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(flattened_fft)

# Reshape patch_fft to match the rank of patch_mfcc (2D)
patch_mfcc = Reshape((-1, projection_dim))(patch_mfcc)
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Concatenate MFCC and FFT patches
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])

# Reshape combined patches for the PVT block
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# Pyramid Vision Transformer (PVT) block
def pvt_block(x, num_channels):
    shortcut = x
    # Projection layers
    x = Dense(num_channels, kernel_initializer=HeNormal())(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)

    x = Dense(num_channels, kernel_initializer=HeNormal())(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)

    # Project the shortcut to match the number of channels
    shortcut = Dense(num_channels * 4, kernel_initializer=HeNormal())(shortcut)
    x = Dense(num_channels * 4, kernel_initializer=HeNormal())(x)
    x = LayerNormalization()(x)

    # Add shortcut and output
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)

    return x

# Apply PVT block
pvt_output = pvt_block(combined_patches, num_channels=256)  # Adjust num_channels as needed

# Vision Transformer (ViT) block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    # Multi-head self-attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    # Feed Forward Network
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

# Apply ViT block to PVT output
vit_output = vit_block(pvt_output)

# Flatten the output before passing to final layers
flattened_vision_output = Flatten()(vit_output)

# Define the inputs for the second pipeline (distillation and multimodal)
input_distillation_second = Input(shape=(512,), name='input_distillation_second')  # Adjusted shape
input_multimodal_second = Input(shape=(512,), name='input_multimodal_second')  # Adjusted shape

# Concatenate the flattened outputs
merged_output = Concatenate()([input_distillation_second,
                               input_multimodal_second,
                               flattened_vision_output])

# Ensure that the merged_output shape is correct
merged_output_shape = merged_output.shape[-1]  # Should match (None, 1536) if concatenation is correct

# Add dense layers after concatenation
flattened_output = Flatten()(merged_output)  # Ensure that the shape becomes (None, 1536)

# Adding L2 regularization and Dropout
dense_layer = Dense(512, activation='relu', kernel_regularizer=l2(1e-5))(flattened_output)
dropout_layer = Dropout(0.5)(dense_layer)  # Change dropout rate as needed
final_output = Dense(2, activation='softmax', name='output')(dropout_layer)

# Compile the model
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)
optimizer = Adam(learning_rate=1e-4)  # Change learning rate as needed
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Model summary to verify shape compatibility
model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_21 (Flatten)      │ (None, 26)             │              0 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_22 (Flatten)      │ (None, 974)            │              0 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_27 (Dense)          │ (None, 256)            │          6,912 │ flatten_21[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_28 (Dense)          │ (None, 256)            │        249,600 │ flatten_22[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_14 (Reshape)      │ (None, 1, 256)         │              0 │ dense_27[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_15 (Reshape)      │ (None, 1, 256)         │              0 │ dense_28[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_12            │ (None, 2, 256)         │              0 │ reshape_14[0][0],      │
│ (Concatenate)             │                        │                │ reshape_15[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_16 (Reshape)      │ (None, 2, 256)         │              0 │ concatenate_12[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_29 (Dense)          │ (None, 2, 256)         │         65,792 │ reshape_16[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_23    │ (None, 2, 256)         │            512 │ dense_29[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_12 (ReLU)           │ (None, 2, 256)         │              0 │ layer_normalization_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_30 (Dense)          │ (None, 2, 256)         │         65,792 │ re_lu_12[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_24    │ (None, 2, 256)         │            512 │ dense_30[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_13 (ReLU)           │ (None, 2, 256)         │              0 │ layer_normalization_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_32 (Dense)          │ (None, 2, 1024)        │        263,168 │ re_lu_13[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_31 (Dense)          │ (None, 2, 1024)        │        263,168 │ reshape_16[0][0]       │
├──────────────────────

 Total params: 12,857,858 (49.05 MB)

 Trainable params: 12,857,858 (49.05 MB)

 Non-trainable params: 0 (0.00 B)

# DENSE LAYERS IN MULTIPLE OF 2

In [ ]:
import os
import cv2
import numpy as np
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# Define input shapes for MFCC and FFT features
input_mfcc_shape = (13, 2)  # MFCC shape: 13 coefficients, 2 time steps (assuming 2 channels)
input_fft_shape = (974,)  # FFT shape: 974 features

# Inputs for MFCC and FFT features
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Flatten the inputs
flattened_mfcc = Flatten()(input_mfcc)
flattened_fft = Flatten()(input_fft)

# Patch Embedding layer for MFCC and FFT
projection_dim = 256  # Adjust as needed
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape patch_fft to match the rank of patch_mfcc
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Concatenate patches
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# Pyramid Vision Transformer (PVT) block
def pvt_block(x, num_channels):
    shortcut = x
    # Projection layers
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    # Project the shortcut to match the number of channels
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    return x

# Apply PVT block
pvt_output = pvt_block(combined_patches, num_channels=256)  # Adjust num_channels as needed

# Vision Transformer (ViT) block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    # Multi-head self-attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    # Feed Forward Network
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

vit_output = vit_block(pvt_output)
vision_transformer_output = Dense(1, activation='sigmoid', name='vision_transformer_output')(tf.keras.layers.Flatten()(vit_output))

# Define the inputs for the second pipeline
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Flatten the vision transformer output before concatenation
flattened_vision_output = Flatten()(vision_transformer_output)

# Concatenate the flattened outputs
merged_output = Concatenate()([input_distillation_second,
                               input_multimodal_second,
                               flattened_vision_output])

# Flatten the merged output before passing it to the Dense layer
flattened_output = Flatten()(merged_output)

# Dense layers with progressive reduction
dense_layer_1 = Dense(128, activation='relu', kernel_regularizer=l2(1e-5))(flattened_output)
dropout_layer_1 = Dropout(0.5)(dense_layer_1)
dense_layer_2 = Dense(64, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)
dense_layer_3 = Dense(32, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_2)
dropout_layer_3 = Dropout(0.5)(dense_layer_3)
dense_layer_4 = Dense(16, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_3)
dropout_layer_4 = Dropout(0.5)(dense_layer_4)
dense_layer_5 = Dense(8, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_4)
dropout_layer_5 = Dropout(0.5)(dense_layer_5)
dense_layer_6 = Dense(4, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_5)
dropout_layer_6 = Dropout(0.5)(dense_layer_6)
dense_layer_7 = Dense(2, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_6)
dropout_layer_7 = Dropout(0.5)(dense_layer_7)

# Final output layer
final_output = Dense(2, activation='softmax', name='output')(dropout_layer_7)

# Define optimizer with custom learning rate
optimizer = Adam(learning_rate=1e-4)  # Change learning rate as needed

# Create model
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)

# Compile the model with the new optimizer
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_3 (Cast)             │ (None, 974)            │              0 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_2 (Cast)             │ (None, 13, 2)          │              0 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_28 (Dense)          │ (None, 256)            │        249,600 │ cast_3[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_27 (Dense)          │ (None, 13, 256)        │            768 │ cast_2[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_2 (Reshape)       │ (None, 1, 256)         │              0 │ dense_28[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_2             │ (None, 14, 256)        │              0 │ dense_27[0][0],        │
│ (Concatenate)             │                        │                │ reshape_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_3 (Reshape)       │ (None, 14, 256)        │              0 │ concatenate_2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_29 (Dense)          │ (None, 14, 256)        │         65,792 │ reshape_3[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_10    │ (None, 14, 256)        │            512 │ dense_29[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_15 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_30 (Dense)          │ (None, 14, 256)        │         65,792 │ re_lu_15[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_11    │ (None, 14, 256)        │            512 │ dense_30[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_16 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_32 (Dense)          │ (None, 14, 1024)       │        263,168 │ re_lu_16[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_31 (Dense)          │ (None, 14, 1024)       │        263,168 │ reshape_3[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_12    │ (None, 14, 1024)       │          2,048 │ dense_32[0][0]         │
│ (LayerNormalization) 

 Total params: 12,078,893 (46.08 MB)

 Trainable params: 12,078,893 (46.08 MB)

 Non-trainable params: 0 (0.00 B)

# DATA GENERATOR FOR LOADING

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


# accuracy when dense layers progressively dec to 2

In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=25,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/25


/usr/local/lib/python3.10/dist-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['input_mfcc', 'input_fft', 'input_distillation_second', 'input_multimodal_second']. Received: the structure of inputs=('*', '*', '*', '*')
  warnings.warn(


392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.6510 - loss: 0.7650Found 1541 images belonging to 2 classes.
392/392 ━━━━━━━━━━━━━━━━━━━━ 3912s 10s/step - accuracy: 0.6511 - loss: 0.7649 - val_accuracy: 0.8353 - val_loss: 0.6710
Epoch 2/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 81s 166ms/step - accuracy: 0.7913 - loss: 0.6774 - val_accuracy: 0.8350 - val_loss: 0.6490
Epoch 3/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 158ms/step - accuracy: 0.8099 - loss: 0.6490 - val_accuracy: 0.8376 - val_loss: 0.6282
Epoch 4/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 61s 156ms/step - accuracy: 0.8141 - loss: 0.6293 - val_accuracy: 0.8330 - val_loss: 0.6105
Epoch 5/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 59s 150ms/step - accuracy: 0.8150 - loss: 0.6119 - val_accuracy: 0.8383 - val_loss: 0.5911
Epoch 6/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 54s 139ms/step - accuracy: 0.8186 - loss: 0.5945 - val_accuracy: 0.8310 - val_loss: 0.5782
Epoch 7/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 55s 142ms/step - accuracy: 0.8212 - loss: 0.5789 - val_accuracy: 0.84

In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=5,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Epoch 1/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 63s 160ms/step - accuracy: 0.8226 - loss: 0.5499 - val_accuracy: 0.8357 - val_loss: 0.5120
Epoch 2/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 160ms/step - accuracy: 0.8176 - loss: 0.5483 - val_accuracy: 0.8337 - val_loss: 0.5042
Epoch 3/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 63s 160ms/step - accuracy: 0.8282 - loss: 0.5247 - val_accuracy: 0.8370 - val_loss: 0.4928
Epoch 4/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 158ms/step - accuracy: 0.8249 - loss: 0.5189 - val_accuracy: 0.8403 - val_loss: 0.4814
Epoch 5/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 60s 152ms/step - accuracy: 0.8166 - loss: 0.5182 - val_accuracy: 0.8304 - val_loss: 0.4848
Epoch 6/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 56s 142ms/step - accuracy: 0.8197 - loss: 0.5121 - val_accuracy: 0.8363 - val_loss: 0.4740
Epoch 7/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 57s 144ms/step - accuracy: 0.8209 - loss: 0.5102 - val_accuracy: 0.8396 - val_loss: 0.4675
Epoch 8/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 58s 149ms/step - accuracy: 0.8151 - loss: 0

Results of the previous model below

In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=15,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step - accuracy: 0.8149 - loss: 0.4800Found 1541 images belonging to 2 classes.
392/392 ━━━━━━━━━━━━━━━━━━━━ 90s 231ms/step - accuracy: 0.8149 - loss: 0.4800 - val_accuracy: 0.8366 - val_loss: 0.4448
Epoch 2/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 66s 154ms/step - accuracy: 0.8284 - loss: 0.4580 - val_accuracy: 0.8337 - val_loss: 0.4519
Epoch 3/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 60s 154ms/step - accuracy: 0.8200 - loss: 0.4713 - val_accuracy: 0.8443 - val_loss: 0.4346
Epoch 4/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 59s 152ms/step - accuracy: 0.8213 - loss: 0.4694 - val_accuracy: 0.8257 - val_loss: 0.4624
Epoch 5/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 57s 146ms/step - accuracy: 0.8207 - loss: 0.4705 - val_accuracy: 0.8383 - val_loss: 0.4434
Epoch 6/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 53s 137ms/step - accuracy: 0.8259 - loss: 0.4625 - val_accuracy: 0.8357 - val_loss: 0.4485
Epoch 7/15
392/392 ━━━━━━━━━━━━━━━━━━━━ 55s 140ms/ste

In [ ]:
#AFTER USING THE NEW MODEL, THESE RESULTS
# HERE DENSE LAYERS HAVE BEEN REMOVED

# DENSE LAYER removed here

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Conv1D, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Define input shapes for MFCC and FFT features
input_mfcc_shape = (13, 2)  # MFCC shape: 13 coefficients, 2 time steps (assuming 2 channels)
input_fft_shape = (974,)  # FFT shape: 974 features

# Inputs for MFCC and FFT features
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Flattening layers
flattened_mfcc = Flatten()(input_mfcc)
flattened_fft = Flatten()(input_fft)

# Patch Embedding layers
projection_dim = 256
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape patch_fft to match the rank of patch_mfcc
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Concatenate patches using Keras Concatenate layer
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# Pyramid Vision Transformer (PVT) block
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT block
pvt_output = pvt_block(combined_patches, num_channels=256)

# Vision Transformer (ViT) block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    # Multi-head self-attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    # Feed Forward Network
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

vit_output = vit_block(pvt_output)

# Flatten the output of ViT block
flattened_vit_output = Flatten()(vit_output)

# Define inputs for additional data (e.g., distillation and multimodal)
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate the flattened outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, flattened_vit_output])

# Flatten the merged output before Dense layer
flattened_output = Flatten()(merged_output)

# Adding L2 regularization and Dropout
dense_layer = Dense(512, activation='relu', kernel_regularizer=l2(1e-5))(flattened_output)
dropout_layer = Dropout(0.5)(dense_layer)
final_output = Dense(2, activation='softmax', name='output')(dropout_layer)

# Compile the model
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)

# Define optimizer with custom learning rate
optimizer = Adam(learning_rate=1e-4)

# Compile the model with the new optimizer
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_42 (Dense)          │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_41 (Dense)          │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_19 (Reshape)      │ (None, 1, 256)         │              0 │ dense_42[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_16            │ (None, 14, 256)        │              0 │ dense_41[0][0],        │
│ (Concatenate)             │                        │                │ reshape_19[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_20 (Reshape)      │ (None, 14, 256)        │              0 │ concatenate_16[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_43 (Dense)          │ (None, 14, 256)        │         65,792 │ reshape_20[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_33    │ (None, 14, 256)        │            512 │ dense_43[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_18 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_44 (Dense)          │ (None, 14, 256)        │         65,792 │ re_lu_18[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_34    │ (None, 14, 256)        │            512 │ dense_44[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_19 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_46 (Dense)          │ (None, 14, 1024)       │        263,168 │ re_lu_19[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_45 (Dense)          │ (None, 14, 1024)       │        263,168 │ reshape_20[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_35    │ (None, 14, 1024)       │          2,048 │ dense_46[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_19 (Add)              │ (None, 14, 1024)       │              0 │ dense_45[0][0],        │
│                           │                        │                │ layer_normalization_3… │
├──────────────────────

 Total params: 13,901,314 (53.03 MB)

 Trainable params: 13,901,314 (53.03 MB)

 Non-trainable params: 0 (0.00 B)

85% wala

In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=25,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Epoch 1/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 157ms/step - accuracy: 0.8234 - loss: 0.4827 - val_accuracy: 0.8270 - val_loss: 0.4685
Epoch 2/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 61s 156ms/step - accuracy: 0.8160 - loss: 0.4940 - val_accuracy: 0.8390 - val_loss: 0.4510
Epoch 3/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 61s 157ms/step - accuracy: 0.8267 - loss: 0.4773 - val_accuracy: 0.8383 - val_loss: 0.4493
Epoch 4/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 60s 154ms/step - accuracy: 0.8157 - loss: 0.4930 - val_accuracy: 0.8370 - val_loss: 0.4508
Epoch 5/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 59s 151ms/step - accuracy: 0.8194 - loss: 0.4888 - val_accuracy: 0.8343 - val_loss: 0.4563
Epoch 6/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 55s 140ms/step - accuracy: 0.8158 - loss: 0.4896 - val_accuracy: 0.8370 - val_loss: 0.4567
Epoch 7/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 55s 141ms/step - accuracy: 0.8204 - loss: 0.4838 - val_accuracy: 0.8370 - val_loss: 0.4526
Epoch 8/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 58s 148ms/step - accuracy: 0.8184 - loss: 0

NEW MODEL WALA DATA AUGMENTATION


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,  # Randomly rotate images in the range (degrees)
    width_shift_range=0.2,  # Randomly shift images horizontally
    height_shift_range=0.2,  # Randomly shift images vertically
    shear_range=0.2,  # Shear angle in counter-clockwise direction
    zoom_range=0.2,  # Randomly zoom inside images
    horizontal_flip=True,  # Randomly flip images horizontally
    fill_mode='nearest'  # Fill in newly created pixels with the nearest value
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=25,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step - accuracy: 0.8005 - loss: 0.7307Found 1541 images belonging to 2 classes.
392/392 ━━━━━━━━━━━━━━━━━━━━ 152s 351ms/step - accuracy: 0.8006 - loss: 0.7303 - val_accuracy: 0.8359 - val_loss: 0.4576
Epoch 2/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 124s 294ms/step - accuracy: 0.8215 - loss: 0.4915 - val_accuracy: 0.8370 - val_loss: 0.4519
Epoch 3/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 115s 294ms/step - accuracy: 0.8200 - loss: 0.4902 - val_accuracy: 0.8350 - val_loss: 0.4586
Epoch 4/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 112s 287ms/step - accuracy: 0.8235 - loss: 0.4834 - val_accuracy: 0.8330 - val_loss: 0.4580
Epoch 5/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 110s 280ms/step - accuracy: 0.8197 - loss: 0.4898 - val_accuracy: 0.8383 - val_loss: 0.4492
Epoch 6/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 109s 278ms/step - accuracy: 0.8171 - loss: 0.4921 - val_accuracy: 0.8357 - val_loss: 0.4554
Epoch 7/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 107s 27

# DATA AUGMENTATION done here to improve and train the model again
ADDED IN DATA GENERATOR CODE

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,         # Normalization
    rotation_range=20,      # Randomly rotate images by 20 degrees
    width_shift_range=0.2,  # Randomly shift images horizontally by 20% of the width
    height_shift_range=0.2, # Randomly shift images vertically by 20% of the height
    shear_range=0.2,        # Shear transformation
    zoom_range=0.2,         # Random zoom
    horizontal_flip=True,   # Randomly flip images horizontally
    fill_mode='nearest'     # Fill in new pixels
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)



Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=10,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 535ms/step - accuracy: 0.8214 - loss: 0.5093Found 1541 images belonging to 2 classes.
392/392 ━━━━━━━━━━━━━━━━━━━━ 225s 574ms/step - accuracy: 0.8214 - loss: 0.5093 - val_accuracy: 0.8359 - val_loss: 0.4951
Epoch 2/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 115s 295ms/step - accuracy: 0.8247 - loss: 0.5020 - val_accuracy: 0.8350 - val_loss: 0.4883
Epoch 3/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 114s 291ms/step - accuracy: 0.8157 - loss: 0.5103 - val_accuracy: 0.8363 - val_loss: 0.4835
Epoch 4/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 114s 292ms/step - accuracy: 0.8180 - loss: 0.5005 - val_accuracy: 0.8390 - val_loss: 0.4725
Epoch 5/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 111s 283ms/step - accuracy: 0.8258 - loss: 0.4888 - val_accuracy: 0.8323 - val_loss: 0.4801
Epoch 6/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 108s 276ms/step - accuracy: 0.8210 - loss: 0.4901 - val_accuracy: 0.8357 - val_loss: 0.4720
Epoch 7/10
392/392 ━━━━━━━━━━━━━━━━━━━━ 108s 27

# pvt block removed

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Conv1D, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Define input shapes for MFCC and FFT features
input_mfcc_shape = (13, 2)  # MFCC shape: 13 coefficients, 2 time steps (assuming 2 channels)
input_fft_shape = (974,)  # FFT shape: 974 features

# Inputs for MFCC and FFT features
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Flattening layers
flattened_mfcc = Flatten()(input_mfcc)
flattened_fft = Flatten()(input_fft)

# Patch Embedding layers
projection_dim = 256
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape patch_fft to match the rank of patch_mfcc
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Concatenate patches using Keras Concatenate layer
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# Vision Transformer (ViT) block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    # Multi-head self-attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    # Feed Forward Network
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

vit_output = vit_block(combined_patches)

# Flatten the output of ViT block
flattened_vit_output = Flatten()(vit_output)

# Define inputs for additional data (e.g., distillation and multimodal)
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate the flattened outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, flattened_vit_output])

# Flatten the merged output before Dense layer
flattened_output = Flatten()(merged_output)

# Adding L2 regularization and Dropout
dense_layer = Dense(512, activation='relu', kernel_regularizer=l2(1e-5))(flattened_output)
dropout_layer = Dropout(0.5)(dense_layer)
final_output = Dense(2, activation='softmax', name='output')(dropout_layer)

# Compile the model
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)

# Define optimizer with custom learning rate
optimizer = Adam(learning_rate=1e-4)

# Compile the model with the new optimizer
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_49 (Dense)          │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_48 (Dense)          │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_21 (Reshape)      │ (None, 1, 256)         │              0 │ dense_49[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_18            │ (None, 14, 256)        │              0 │ dense_48[0][0],        │
│ (Concatenate)             │                        │                │ reshape_21[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_22 (Reshape)      │ (None, 14, 256)        │              0 │ concatenate_18[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention_7    │ (None, 14, 256)        │      2,103,552 │ reshape_22[0][0],      │
│ (MultiHeadAttention)      │                        │                │ reshape_22[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_28 (Dropout)      │ (None, 14, 256)        │              0 │ multi_head_attention_… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_22 (Add)              │ (None, 14, 256)        │              0 │ dropout_28[0][0],      │
│                           │                        │                │ reshape_22[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_38    │ (None, 14, 256)        │            512 │ add_22[0][0]           │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_27 (Conv1D)        │ (None, 14, 512)        │        393,728 │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_28 (Conv1D)        │ (None, 14, 512)        │        786,944 │ conv1d_27[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_29 (Conv1D)        │ (None, 14, 256)        │        131,328 │ conv1d_28[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_29 (Dropout)      │ (None, 14, 256)        │              0 │ conv1d_29[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_30 (Conv1D)        │ (None, 14, 256)        │         65,792 │ layer_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_23 (Add)              │ (None, 14, 256)        │              0 │ dropout_29[0][0],      │
│                           │                        │                │ conv1d_30[0][0]        │
├──────────────────────

 Total params: 5,570,306 (21.25 MB)

 Trainable params: 5,570,306 (21.25 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))

# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, width_shift_range=0.2, height_shift_range=0.2, shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest')
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=25,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)


Epoch 1/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 113s 288ms/step - accuracy: 0.8198 - loss: 0.4859 - val_accuracy: 0.8357 - val_loss: 0.4592
Epoch 2/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 113s 288ms/step - accuracy: 0.8223 - loss: 0.4808 - val_accuracy: 0.8363 - val_loss: 0.4523
Epoch 3/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 112s 286ms/step - accuracy: 0.8241 - loss: 0.4797 - val_accuracy: 0.8343 - val_loss: 0.4586
Epoch 4/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 111s 285ms/step - accuracy: 0.8192 - loss: 0.4855 - val_accuracy: 0.8376 - val_loss: 0.4497
Epoch 5/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 109s 280ms/step - accuracy: 0.8233 - loss: 0.4795 - val_accuracy: 0.8363 - val_loss: 0.4552
Epoch 6/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 108s 275ms/step - accuracy: 0.8203 - loss: 0.4835 - val_accuracy: 0.8350 - val_loss: 0.4536
Epoch 7/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 108s 276ms/step - accuracy: 0.8252 - loss: 0.4770 - val_accuracy: 0.8304 - val_loss: 0.4610
Epoch 8/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 110s 280ms/step - accuracy: 0.8211 -

# ALL DENSE LAYERS REMOVED HERE

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Conv1D, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Define input shapes for MFCC and FFT features
input_mfcc_shape = (13, 2)  # MFCC shape: 13 coefficients, 2 time steps (assuming 2 channels)
input_fft_shape = (974,)  # FFT shape: 974 features

# Inputs for MFCC and FFT features
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Flattening layers
flattened_mfcc = Flatten()(input_mfcc)
flattened_fft = Flatten()(input_fft)

# Patch Embedding layers
projection_dim = 256
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape patch_fft to match the rank of patch_mfcc
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Concatenate patches using Keras Concatenate layer
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# Pyramid Vision Transformer (PVT) block
def pvt_block(x, num_channels):
    shortcut = x
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)
    return x

# Apply PVT block
pvt_output = pvt_block(combined_patches, num_channels=256)

# Vision Transformer (ViT) block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    # Multi-head self-attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    # Feed Forward Network
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

vit_output = vit_block(pvt_output)

# Flatten the output of ViT block
flattened_vit_output = Flatten()(vit_output)

# Define inputs for additional data (e.g., distillation and multimodal)
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Concatenate the flattened outputs
merged_output = Concatenate()([input_distillation_second, input_multimodal_second, flattened_vit_output])

# Adding L2 regularization and Dropout
dropout_layer = Dropout(0.5)(merged_output)
final_output = Dense(2, activation='softmax', name='output')(dropout_layer)

# Compile the model
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)

# Define optimizer with custom learning rate
optimizer = Adam(learning_rate=1e-4)

# Compile the model with the new optimizer
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_52 (Dense)          │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_51 (Dense)          │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_23 (Reshape)      │ (None, 1, 256)         │              0 │ dense_52[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_20            │ (None, 14, 256)        │              0 │ dense_51[0][0],        │
│ (Concatenate)             │                        │                │ reshape_23[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_24 (Reshape)      │ (None, 14, 256)        │              0 │ concatenate_20[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_53 (Dense)          │ (None, 14, 256)        │         65,792 │ reshape_24[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_40    │ (None, 14, 256)        │            512 │ dense_53[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_21 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_4… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_54 (Dense)          │ (None, 14, 256)        │         65,792 │ re_lu_21[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_41    │ (None, 14, 256)        │            512 │ dense_54[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_22 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_4… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_56 (Dense)          │ (None, 14, 1024)       │        263,168 │ re_lu_22[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_55 (Dense)          │ (None, 14, 1024)       │        263,168 │ reshape_24[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_42    │ (None, 14, 1024)       │          2,048 │ dense_56[0][0]         │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_24 (Add)              │ (None, 14, 1024)       │              0 │ dense_55[0][0],        │
│                           │                        │                │ layer_normalization_4… │
├──────────────────────

 Total params: 12,070,918 (46.05 MB)

 Trainable params: 12,070,918 (46.05 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))

# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, width_shift_range=0.2, height_shift_range=0.2, shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest')
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=25,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Found 12570 images belonging to 2 classes.
Epoch 1/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 0s 555ms/step - accuracy: 0.8030 - loss: 2.5569Found 1541 images belonging to 2 classes.
392/392 ━━━━━━━━━━━━━━━━━━━━ 242s 580ms/step - accuracy: 0.8030 - loss: 2.5577 - val_accuracy: 0.8359 - val_loss: 2.6444
Epoch 2/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 126s 296ms/step - accuracy: 0.8267 - loss: 2.7937 - val_accuracy: 0.8337 - val_loss: 2.6810
Epoch 3/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 113s 288ms/step - accuracy: 0.8170 - loss: 2.9497 - val_accuracy: 0.8376 - val_loss: 2.6169
Epoch 4/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 112s 286ms/step - accuracy: 0.8154 - loss: 2.9753 - val_accuracy: 0.8416 - val_loss: 2.5528
Epoch 5/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 109s 278ms/step - accuracy: 0.8250 - loss: 2.8210 - val_accuracy: 0.8304 - val_loss: 2.7344
Epoch 6/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 106s 272ms/step - accuracy: 0.8174 - loss: 2.9424 - val_accuracy: 0.8363 - val_loss: 2.6383
Epoch 7/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 108s 27

Testing results OF OLD MODEL

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))

# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for normalization
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create safe test generator
safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=32,
    class_mode='binary'
)

# Combined test generator
combined_test_gen = combined_generator(safe_test_generator, distillation_predictions_test, multimodal_predictions_test, batch_size=32)

# Use the modified generator for testing
combined_test_gen = modify_generator_output(combined_test_gen)

# Calculate steps for testing using len(generator.filenames)
test_steps = len(test_generator.filenames) // test_generator.batch_size

# Model evaluation
test_loss, test_accuracy = model.evaluate(
    combined_test_gen,
    steps=test_steps
)

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

Found 992 images belonging to 2 classes.
31/31 ━━━━━━━━━━━━━━━━━━━━ 65s 2s/step - accuracy: 0.7491 - loss: 0.5794
Test Loss: 0.5887599587440491
Test Accuracy: 0.742943525314331


In [ ]:
import os
import cv2
import numpy as np
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Reshape, Add, MultiHeadAttention, Dropout, Concatenate, Flatten, Conv1D, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# Define input shapes for MFCC and FFT features
input_mfcc_shape = (13, 2)  # MFCC shape: 13 coefficients, 2 time steps (assuming 2 channels)
input_fft_shape = (974,)  # FFT shape: 974 features

# Inputs for MFCC and FFT features
input_mfcc = Input(shape=input_mfcc_shape, name='input_mfcc')
input_fft = Input(shape=input_fft_shape, name='input_fft')

# Patch Embedding layer for MFCC and FFT
projection_dim = 256  # Adjust as needed
patch_mfcc = Dense(projection_dim, kernel_initializer=HeNormal())(input_mfcc)
patch_fft = Dense(projection_dim, kernel_initializer=HeNormal())(input_fft)

# Reshape patch_fft to match the rank of patch_mfcc
patch_fft = Reshape((-1, projection_dim))(patch_fft)

# Concatenate patches
combined_patches = Concatenate(axis=1)([patch_mfcc, patch_fft])
combined_patches = Reshape((-1, projection_dim))(combined_patches)

# Pyramid Vision Transformer (PVT) block
def pvt_block(x, num_channels):
    shortcut = x
    # Projection layers
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    x = Dense(num_channels)(x)
    x = LayerNormalization()(x)
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    # Project the shortcut to match the number of channels
    shortcut = Dense(num_channels * 4)(shortcut)
    x = Dense(num_channels * 4)(x)
    x = LayerNormalization()(x)
    x = Add()([shortcut, x])
    x = tf.keras.activations.relu(x)  # Use Keras ReLU
    return x

# Apply PVT block
pvt_output = pvt_block(combined_patches, num_channels=256)  # Adjust num_channels as needed

# Vision Transformer (ViT) block
def vit_block(x, num_heads=8, ff_dim=512, dropout_rate=0.1):
    # Multi-head self-attention
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(attn_output + x)

    # Feed Forward Network
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(out1)
    ffn = Conv1D(filters=ff_dim, kernel_size=3, activation='relu', padding='same')(ffn)
    ffn = Conv1D(filters=projection_dim, kernel_size=1)(ffn)
    ffn = Dropout(dropout_rate)(ffn)

    out1 = Conv1D(filters=projection_dim, kernel_size=1)(out1)
    out2 = LayerNormalization(epsilon=1e-6)(ffn + out1)
    return out2

vit_output = vit_block(pvt_output)
vision_transformer_output = Dense(1, activation='sigmoid', name='vision_transformer_output')(tf.keras.layers.Flatten()(vit_output))

# Define the inputs for the second pipeline
input_distillation_second = Input(shape=(distillation_predictions_train.shape[1],), name='input_distillation_second')
input_multimodal_second = Input(shape=(multimodal_predictions_train.shape[1],), name='input_multimodal_second')

# Reshape or adjust inputs if needed
reshaped_distillation = Dense(256, activation='relu')(input_distillation_second)
reshaped_multimodal = Dense(256, activation='relu')(input_multimodal_second)

# Concatenate the features from different sources
merged_features = Concatenate()([reshaped_distillation, reshaped_multimodal, Flatten()(vision_transformer_output)])

# Ensure merged_features is 3D for GlobalAveragePooling1D
merged_features = Reshape((1, -1))(merged_features)  # Reshape to (batch_size, sequence_length, features)

# Use pooling layers if appropriate
merged_features = GlobalAveragePooling1D()(merged_features)

# Dense layers with progressive reduction
dense_layer_1 = Dense(128, activation='relu', kernel_regularizer=l2(1e-5))(merged_features)
dropout_layer_1 = Dropout(0.5)(dense_layer_1)
dense_layer_2 = Dense(64, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_1)
dropout_layer_2 = Dropout(0.5)(dense_layer_2)
dense_layer_3 = Dense(32, activation='relu', kernel_regularizer=l2(1e-5))(dropout_layer_2)

# Final output layer
final_output = Dense(2, activation='softmax', name='output')(dense_layer_3)

# Define optimizer with custom learning rate
optimizer = Adam(learning_rate=1e-4)  # Adjust learning rate as needed

# Create model
model = Model(inputs=[input_mfcc, input_fft, input_distillation_second, input_multimodal_second], outputs=final_output)

# Compile the model with the new optimizer
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_fft (InputLayer)    │ (None, 974)            │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_mfcc (InputLayer)   │ (None, 13, 2)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_275 (Dense)         │ (None, 256)            │        249,600 │ input_fft[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_274 (Dense)         │ (None, 13, 256)        │            768 │ input_mfcc[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_67 (Reshape)      │ (None, 1, 256)         │              0 │ dense_275[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_66            │ (None, 14, 256)        │              0 │ dense_274[0][0],       │
│ (Concatenate)             │                        │                │ reshape_67[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_68 (Reshape)      │ (None, 14, 256)        │              0 │ concatenate_66[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_276 (Dense)         │ (None, 14, 256)        │         65,792 │ reshape_68[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_143   │ (None, 14, 256)        │            512 │ dense_276[0][0]        │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_84 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_277 (Dense)         │ (None, 14, 256)        │         65,792 │ re_lu_84[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_144   │ (None, 14, 256)        │            512 │ dense_277[0][0]        │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_85 (ReLU)           │ (None, 14, 256)        │              0 │ layer_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_279 (Dense)         │ (None, 14, 1024)       │        263,168 │ re_lu_85[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_278 (Dense)         │ (None, 14, 1024)       │        263,168 │ reshape_68[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization_145   │ (None, 14, 1024)       │          2,048 │ dense_279[0][0]        │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_85 (Add)              │ (None, 14, 1024)       │              0 │ dense_278[0][0],       │
│                           │                        │                │ layer_normalization_1… │
├──────────────────────

 Total params: 12,144,547 (46.33 MB)

 Trainable params: 12,144,547 (46.33 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
import os

# Enable mixed precision training
tf.keras.mixed_precision.set_global_policy('mixed_float16')

def combined_generator(image_gen, distillation_data, multimodal_data, batch_size):
    while True:
        try:
            image_data, image_labels = next(image_gen)
        except Exception as e:
            print(f"Error loading image batch: {e}")
            continue

        actual_batch_size = image_data.shape[0]
        mfcc_data = np.zeros((actual_batch_size, 13, 2))
        fft_data = np.zeros((actual_batch_size, 974))
        image_labels = tf.keras.utils.to_categorical(image_labels, num_classes=2)

        if distillation_data.size > 0:
            start_idx_distill = np.random.randint(0, len(distillation_data) - actual_batch_size)
            batch_distillation_data = distillation_data[start_idx_distill: start_idx_distill + actual_batch_size]
        else:
            batch_distillation_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        if multimodal_data.size > 0:
            start_idx_multi = np.random.randint(0, len(multimodal_data) - actual_batch_size)
            batch_multimodal_data = multimodal_data[start_idx_multi: start_idx_multi + actual_batch_size]
        else:
            batch_multimodal_data = np.empty((actual_batch_size, 0))  # Ensure matching batch size

        yield ([mfcc_data, fft_data, batch_distillation_data, batch_multimodal_data], image_labels)

def modify_generator_output(generator):
    for inputs, targets in generator:
        yield (tuple(tf.convert_to_tensor(arr, dtype=tf.float32) for arr in inputs),
               tf.convert_to_tensor(targets, dtype=tf.float32))



# Define the base directory for your data
base_dir = '/content/drive/MyDrive/Celeb-DF/Split-Frames'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Define ImageDataGenerator for data augmentation and normalization
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Custom function to skip corrupt images
def safe_flow_from_directory(datagen, directory, *args, **kwargs):
    iterator = datagen.flow_from_directory(directory, *args, **kwargs)
    while True:
        try:
            yield next(iterator)
        except Exception as e:
            print(f"Corrupt image file found: {e}")
            continue  # Skip this batch and move to the next

# Create generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Create safe generators
safe_train_generator = safe_flow_from_directory(
    train_datagen,
    train_dir,
    target_size=(150, 150),  # Adjust based on your model input size
    batch_size=batch_size,
    class_mode='binary'
)

safe_val_generator = safe_flow_from_directory(
    val_datagen,
    val_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

safe_test_generator = safe_flow_from_directory(
    test_datagen,
    test_dir,
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='binary'
)

# Calculate steps per epoch using len(generator.filenames)
train_steps_per_epoch = len(train_generator.filenames) // train_generator.batch_size
val_steps_per_epoch = len(val_generator.filenames) // val_generator.batch_size

# Combined training generator
combined_train_gen = combined_generator(safe_train_generator, distillation_predictions_train, multimodal_predictions_train, batch_size)

# Combined validation generator
combined_val_gen = combined_generator(safe_val_generator, distillation_predictions_test, multimodal_predictions_test, batch_size)

Found 12570 images belonging to 2 classes.
Found 1541 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [ ]:
# Use the modified generator for training and validation
combined_train_gen = modify_generator_output(combined_train_gen)
combined_val_gen = modify_generator_output(combined_val_gen)

# Model training
model.fit(
    combined_train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=25,
    validation_data=combined_val_gen,
    validation_steps=val_steps_per_epoch
)

Epoch 1/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 64s 162ms/step - accuracy: 0.8155 - loss: 0.4869 - val_accuracy: 0.8323 - val_loss: 0.4572
Epoch 2/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 64s 163ms/step - accuracy: 0.8193 - loss: 0.4793 - val_accuracy: 0.8383 - val_loss: 0.4461
Epoch 3/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 63s 162ms/step - accuracy: 0.8215 - loss: 0.4757 - val_accuracy: 0.8350 - val_loss: 0.4529
Epoch 4/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 62s 157ms/step - accuracy: 0.8186 - loss: 0.4804 - val_accuracy: 0.8343 - val_loss: 0.4522
Epoch 5/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 60s 153ms/step - accuracy: 0.8132 - loss: 0.4884 - val_accuracy: 0.8370 - val_loss: 0.4479
Epoch 6/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 56s 143ms/step - accuracy: 0.8233 - loss: 0.4732 - val_accuracy: 0.8363 - val_loss: 0.4478
Epoch 7/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 56s 143ms/step - accuracy: 0.8200 - loss: 0.4782 - val_accuracy: 0.8416 - val_loss: 0.4418
Epoch 8/25
392/392 ━━━━━━━━━━━━━━━━━━━━ 59s 151ms/step - accuracy: 0.8225 - loss: 0